## Example 2 (Nonlinear): Spacecraft Attitude Control

In [ ]:
## Activate Project Environment
using Pkg

# Robustly find the notebook directory (handles VS Code, Jupyter, terminal)
NOTEBOOK_DIR = let
    d = @__DIR__
    if !isempty(d) && isdir(d)
        d
    elseif isfile(joinpath(pwd(), "Example_2.ipynb"))
        pwd()
    elseif isfile(joinpath(pwd(), "Example_2", "Example_2.ipynb"))
        joinpath(pwd(), "Example_2")
    else
        @warn "Could not detect notebook directory. Using pwd(): $(pwd())"
        pwd()
    end
end

Pkg.activate(NOTEBOOK_DIR)

# If Project.toml is missing or empty, install all required packages
if !isfile(joinpath(NOTEBOOK_DIR, "Project.toml")) || isempty(Pkg.project().dependencies)
    Pkg.add([
        "JuMP", "HiGHS", "Ipopt", "DynamicPolynomials",
        "DifferentialEquations", "Rotations", "Trapz",
        "MatrixEquations", "SumOfSquares", "MosekTools",
        "LaTeXStrings", "PyPlot",
    ])
end

Pkg.instantiate()


In [ ]:
## Import Packages
using JuMP
using HiGHS
using Ipopt
using LinearAlgebra
using PyPlot
using DynamicPolynomials
using DifferentialEquations
using Rotations
using Trapz
using MatrixEquations

using SumOfSquares
using MosekTools

using LaTeXStrings
import PyPlot
const plt = PyPlot

In [ ]:
## Plot Settings
SMALL_SIZE = 7
MEDIUM_SIZE = 8
BIGGER_SIZE = 9

# Set all parameters using rc()
plt.rc("text", usetex=false)
plt.rc("font", family="Times New Roman", size=MEDIUM_SIZE)
plt.rc("mathtext", fontset="cm")
plt.rc("lines", linewidth=1)
plt.rc("legend", labelspacing=0.25, fontsize=MEDIUM_SIZE)
plt.rc("axes", titlesize=MEDIUM_SIZE, labelsize=MEDIUM_SIZE)
plt.rc("xtick", labelsize=SMALL_SIZE)
plt.rc("ytick", labelsize=SMALL_SIZE)
plt.rc("figure", titlesize=BIGGER_SIZE)

In [ ]:
## Attitude Conversions
# Function: euler_to_rps
# Convert Euler angles (in 'zyx' order, as yaw, pitch, roll) to Rodrigues Parameters (RPs)
function euler_to_rps(theta)
    r = RotXYZ(theta[3], theta[2], theta[1])
    rp = RodriguesParam(r)
    return [rp.x, rp.y, rp.z]
end

# Function: rps_to_euler
# Convert Rodrigues Parameters (RPs) to Euler angles (in 'zyx' order, as yaw, pitch, roll)
function rps_to_euler(sigma)
    rp = RodriguesParam(sigma[1], sigma[2], sigma[3])
    r = RotXYZ(rp)
    return [r.theta3, r.theta2, r.theta1]
end

In [ ]:
## Spacecraft Dynamics (for DifferentialEquations.jl)
# x = [sigma, w, h]
# sigma = [sigma1, sigma2, sigma3]

function spacecraft_plant_closure(u_current, params)
    return (x, p, t) -> spacecraft_plant(x, u_current, params)
end

function spacecraft_plant(x, u, params)
    # Extract parameters
    J = params.J

    # States
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Define system parameters
    sigmac = [ 0.0     -sigma[3]  sigma[2]; 
               sigma[3]  0.0     -sigma[1]; 
              -sigma[2]  sigma[1]  0.0    ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')
    
    wc = [ 0.0    -w[3]   w[2]; 
           w[3]    0.0   -w[1]; 
          -w[2]    w[1]   0.0 ]
    
    # Dynamics
    fx = vcat(M_sigma * w,
              inv(J) * (-wc * (J * w + hw)),
              zeros(3))

    gx = vcat(zeros(3, 3),
              inv(J),
              -I(3))
    
    # Define nonlinear control affine form
    xdot = fx + gx * u
    
    return xdot
end

In [ ]:
###########################################
# Helper: integral of a monomial over a box
###########################################

function integrate_monomial_over_bounds(monom, vars, bounds)

    # Compute ∫ monom(x) dx over a hyper-rectangle.

    # vars   - vector of variables [x1, x2, ..., xn]
    # bounds - vector of (a, b) tuples, same order as vars

    @assert length(vars) == length(bounds) "vars and bounds must have same length"

    # Start from the monomial coefficient
    val = DynamicPolynomials.coefficient(monom)

    # Multiply contributions from each variable
    for (j, v) in enumerate(vars)
        deg = DynamicPolynomials.degree(monom, v)
        a, b = bounds[j]
        if deg == 0
            # ∫ 1 dx = (b - a)
            val *= (b - a)
        else
            # ∫ x^deg dx = (b^(deg+1) - a^(deg+1)) / (deg+1)
            val *= (b^(deg + 1) - a^(deg + 1)) / (deg + 1)
        end
    end

    return val
end

In [ ]:
## System and Simulation Parameters
t_span = (0.0, 30.0)                           # Simulation time interval [s]
dt = 0.1                                       # Integration time step [s]
J = [1.8140  -0.1185   0.0275;                 # Spacecraft inertia matrix [kg·m²]
    -0.1185   1.7350   0.0169; 
     0.0275   0.0169   3.4320]
u_max = 0.123;                                 # Max control torque per axis [N·m]
u_min = -0.123;                                # Min control torque per axis [N·m]
h_max = 0.4;                                   # Max reaction wheel momentum per axis [N·m·s]
h_min = -0.4;                                  # Min reaction wheel momentum per axis [N·m·s]
umin = u_min * ones(3);                        # Lower torque bound vector (3-axis)
umax = u_max * ones(3);                        # Upper torque bound vector (3-axis)
hmin = h_min * ones(3);                        # Lower momentum bound vector (3-axis)
hmax = h_max * ones(3);                        # Upper momentum bound vector (3-axis)

# Define initial states
theta = [80.0, -30.0, 60.0] * (π / 180.0);     # Initial 3-2-1 Euler angles [rad]
sigmai = euler_to_rps(theta);                  # Initial Rodrigues Parameters
wi = [0.0, 0.0, 0.0];                          # Initial angular velocity [rad/s]
hi = [0.0, 0.0, 0.0];                          # Initial reaction wheel momentum [N·m·s]
x0 = [sigmai; wi; hi];                         # Full initial state vector

# ============================================
# Pointing constraint parameters
# ============================================
b_sensors = [
    [0.0, 0.0, 1.0]                            # Sensor boresight in body frame (along +Z body axis)
]

sigma_target = [0.10, -0.25, 0.0]              # RP target near trajectory midpoint
sigmac_t = [ 0.0           -sigma_target[3]  sigma_target[2];   # Skew-symmetric matrix of sigma_target
             sigma_target[3]  0.0            -sigma_target[1]; 
            -sigma_target[2]  sigma_target[1]  0.0           ]
R_t = (1 / (1 + sigma_target' * sigma_target)) *               # DCM from sigma_target RP
      ((1 - sigma_target' * sigma_target) * I(3) + 2 * (sigma_target * sigma_target') - 2 * sigmac_t)

n_bright = [normalize(R_t' * [0.0, 0.0, 1.0])]                  # Bright-object direction in inertial frame
theta_fov = [deg2rad(15.0)]                                     # Keep-out half-angle cone [rad]

alpha_CBF = 10.0                            # Case 1 CBF class-K gain
alpha1_HOCBF = 1.0                          # Case 2 HOCBF 1st class-K gain
alpha2_HOCBF = 1.0                          # Case 2 HOCBF 2nd class-K gain

# ============================================
# Assemble parameters struct
# ============================================
params = (
    J = J,                                     # Inertia matrix
    u_max = u_max,                             # Scalar torque upper bound
    u_min = u_min,                             # Scalar torque lower bound
    h_max = h_max,                             # Scalar momentum upper bound
    h_min = h_min,                             # Scalar momentum lower bound
    umin = umin,                               # 3-axis torque lower bounds
    umax = umax,                               # 3-axis torque upper bounds
    hmin = hmin,                               # 3-axis momentum lower bounds
    hmax = hmax,                               # 3-axis momentum upper bounds
    b_sensors = b_sensors,                     # Sensor boresight vectors
    n_bright = n_bright,                       # Bright-object inertial directions
    theta_fov = theta_fov,                     # Keep-out cone half-angles
    alpha_CBF = alpha_CBF,                     # CBF class-K gain (Case 1)
    alpha1_HOCBF = alpha1_HOCBF,               # HOCBF 1st class-K gain (Case 2)
    alpha2_HOCBF = alpha2_HOCBF                # HOCBF 2nd class-K gain (Case 2)
);

# Cost function parameters 
Q = I(6) * 1.0;                               # State weighting matrix (6×6 identity)
R = I(3) * 1.0;                               # Control weighting matrix (3×3 identity)

In [ ]:
#####################################################
# SOS policy iteration with S-procedure
#####################################################

function run_SOS_policy_iteration(;
    max_iter::Int = 30,                # Maximum policy iterations
    max_degree::Int = 4,               # Max degree for V

    # Cost function parameters 
    Q = Q,                  # State cost matrix (6×6 identity) for [sigma; w]
    R = R,                  # Input cost matrix (3×3 identity) for control torques

    # Integration domain (state space bounds for integration)
    sigma_scale = 1.0,
    w_scale = 1.0,
    bounds = [
        (-1.0*sigma_scale, 1.0*sigma_scale),                   # Bounds for sigma1
        (-1.0*sigma_scale, 1.0*sigma_scale),                   # Bounds for sigma2
        (-1.0*sigma_scale, 1.0*sigma_scale),                   # Bounds for sigma3

        (-1.0*w_scale, 1.0*w_scale),                   # Bounds for w1
        (-1.0*w_scale, 1.0*w_scale),                   # Bounds for w2
        (-1.0*w_scale, 1.0*w_scale)                    # Bounds for w3
        ], 

    params = params,

    V_diff_tol = 1e-3,               # Convergence tolerance

    verbose::Bool = true,
)

    # Polynomial variable
    @polyvar sigma1 sigma2 sigma3 w1 w2 w3
    vars = [sigma1, sigma2, sigma3, w1, w2, w3]
    sigma = vars[1:3]
    w = vars[4:6]

    # States
    x = vcat(sigma, w)
    n_vars = length(vars) 

    # Initialize the optimal value function and optimal control
    V_opt = []
    u_opt = []

    # Fixed initial controller u(x) = -K_sigma*sigma - K_w*w
    u_init = -1 * sigma - 3 * w

    push!(u_opt, u_init)

    # System dynamics
    sigmac = [ 0.0       -sigma3  sigma2; 
               sigma3  0.0       -sigma1; 
              -sigma2  sigma1  0.0 ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')

    wc = [ 0.0     -w3   w2; 
           w3   0.0     -w1; 
          -w2   w1   0.0 ]

    J = params.J
    J_inv = inv(J)

    fx = vcat(M_sigma * w,
              J_inv * (-wc * (J * w)))

    gx = vcat(zeros(3, 3),
              J_inv)

    # Define nonlinear control affine form: xdot = f(x) + g(x) u
    # xdot = fx + gx * u

    # Define the running state cost function q(x)
    q = x' * Q * x

    # Define the running control cost function r(u)
    R_inv = inv(R)

    # Degree range for V – use 2:degree so V(0) = 0
    value_degrees = 2:2:max_degree
    monoms = DynamicPolynomials.monomials(vars, value_degrees)

    num_monoms = length(monoms)
    
    integral_coeffs = [integrate_monomial_over_bounds(monom, vars, bounds) for monom in monoms]

    g_constraints = []
    for i in 1:n_vars
        # Constraint: 1 - x_i^2 >= 0
        g_i = 1 - vars[i]^2
        push!(g_constraints, g_i)
    end
    n_constraints = length(g_constraints)

    for iter in 1:max_iter
        # SOS model
        model = Model(MosekTools.Optimizer)
        set_silent(model)
        
        # Decision variables: coefficients of V
        @variable(model, p[1:num_monoms])
        
        # S-procedure multipliers for positive definiteness constraint
        @variable(model, p_pd[1:num_monoms,1:n_constraints])
        
        # S-procedure multipliers for Lyapunov derivative constraint
        @variable(model, p_lyap[1:num_monoms,1:n_constraints])
        
        # Construct V
        V_new = sum(p[i] * monoms[i] for i in 1:num_monoms)
        λ_pd = [ sum(p_pd[j, i] * monoms[j] for j in 1:num_monoms) for i in 1:n_constraints ]
        λ_lyap = [ sum(p_lyap[j, i] * monoms[j] for j in 1:num_monoms) for i in 1:n_constraints ]

        for i in 1:n_constraints
            @constraint(model, λ_pd[i] in SOSCone())
            @constraint(model, λ_lyap[i] in SOSCone())
        end

        pd_expr = V_new - sum(λ_pd[i] * g_constraints[i] for i in 1:n_constraints)
        @constraint(model, pd_expr in SOSCone())
        
        # Lyapunov derivative constraint with S-procedure
        grad_V_new = [DynamicPolynomials.differentiate(V_new, v) for v in vars]
        dyn    = fx + gx * u_opt[iter]
        LV_new    = grad_V_new' * dyn
        
        # L(V_new, u_opt[iter]) = -dV_new/dt - stage_cost >= 0
        L_expr = - LV_new - q - u_opt[iter]' * R * u_opt[iter]
        
        # Apply S-procedure: L(x) - Σμ_i*g_i(x) >= 0 for all x in domain
        L_with_sprocedure = L_expr - sum(λ_lyap[i] * g_constraints[i] for i in 1:n_constraints)
        @constraint(model, L_with_sprocedure in SOSCone())
        
        # Objective: minimize ∫_Ω V0(x) dx
        @objective(model, Min, sum(integral_coeffs[i] * p[i] for i in 1:num_monoms))
        
        optimize!(model)
        status = termination_status(model)
            
        if verbose
            println("\n----------------- Iteration $iter -----------------")
            println("Value function V design status = $status")
            println("Max degree of monomials in V: $max_degree")
            println("Number of monomials in V: $num_monoms")
            println("Number of domain constraints: $n_constraints")
            println("Objective value: ", objective_value(model))
            # println("-----------------------------------------------")
        end

        # Extract numeric coefficients and create numeric polynomial
        V_coeffs_opt = value.(p)                                  # numeric vector
        V_new_numeric = sum(V_coeffs_opt[i] * monoms[i] for i in 1:num_monoms)
        push!(V_opt, V_new_numeric)

        # Step 2 – Policy Improvement using numeric polynomial
        grad_V_new_numeric = [DynamicPolynomials.differentiate(V_new_numeric, v) for v in vars]
        u_new = - 0.5 * R_inv * gx' * grad_V_new_numeric
        push!(u_opt, u_new)
        
        if iter > 1
            V_prev = V_opt[iter-1]
            # Convergence check (coefficient norm)
            V_prev_coeffs = [DynamicPolynomials.coefficient(V_prev, monom) for monom in monoms]
            V_diff = norm(V_coeffs_opt - V_prev_coeffs)
            
            if verbose
                println("Change in V coefficients: $V_diff")
            end

            if status == MOI.INFEASIBLE
                if verbose
                    println("\n$status => Optimization problem terminated\n")
                end
                return (
                    status = status,
                )
            end

            if iter == max_iter
                if verbose
                    println("\nMaximum iterations is reached\n")
                end
                return (
                    V = V_new_numeric,
                    u = u_new,
                    V_history = V_opt,
                    u_history = u_opt,
                    iterations = iter,
                    status = status,
                )
            end

            if V_diff < V_diff_tol
                if verbose
                    println("\nConverged after $iter iterations\n")
                end
                return (
                    V = V_new_numeric,
                    u = u_new,
                    V_history = V_opt,
                    u_history = u_opt,
                    iterations = iter,
                    status = status,
                )
            end
        end
    end
end

###################
# Run the algorithm
###################

@time results = run_SOS_policy_iteration()

if results.status !== MOI.INFEASIBLE
    println("\n=== Final Results Summary ===")
    println("Final V*(x) = ", results.V)
    println("Final u*(x) = ", results.u)
end

## Case 1: Reaction wheel momentum constraints

In [ ]:
## GHJBCBF Controller with RW Momentum Constraints (Case 1)
function GHJBCBF_control_case1(t, x, t_span_end, R, params)
    # Define state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]
    
    # Compute V and its gradient (only depends on sigma and w)
    V_sym = results.V
    vars = DynamicPolynomials.variables(V_sym)
    
    # Compute the gradient of V with respect to state variables [sigma; w]
    gradV_sym = [DynamicPolynomials.differentiate(V_sym, v) for v in vars]
    gradV_num = DynamicPolynomials.subs(gradV_sym, vars => x[1:6])
    gradV_num = first.(DynamicPolynomials.coefficients.(gradV_num))
    
    # Define system dynamics for 9-state model
    J = params.J
    J_inv = inv(J)
    
    # Kinematics matrix for RPs
    sigmac = [0.0      -sigma[3]  sigma[2]; 
              sigma[3]  0.0      -sigma[1]; 
             -sigma[2]  sigma[1]  0.0]
    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')
    
    # Cross-product matrices
    wc = [0.0    -w[3]   w[2]; 
          w[3]    0.0   -w[1]; 
         -w[2]    w[1]   0.0]
    
    # Full 9-state dynamics: xdot = fx + gx*u
    fx = vcat(
        M_sigma * w,
        J_inv * (-wc * (J * w + hw)),   # Include hw coupling
        zeros(3)                        # RW drift dynamics
    )
    
    gx = vcat(
        zeros(3, 3),
        J_inv,
        -I(3)                           # RW control matrix
    )
    
    # For gradient computation, we only need the [sigma; w] portion
    gx_reduced = gx[1:6, :]  # First 6 rows for [sigma; w] dynamics
    
    # ============================================
    # Control Barrier Function (CBF) Constraints
    # ============================================
    # CBF for RW momentum: h_min ≤ hw ≤ h_max
    
    # Define barrier functions:
    # B_lower(x) = hw - h_min
    # B_upper(x) = h_max - hw
    
    B_lower = hw - params.hmin  # Element-wise: hw[i] - h_min[i]
    B_upper = params.hmax - hw  # Element-wise: h_max[i] - hw[i]
    
    # From paper equations:
    # Lf B_lower = 0, Lg B_lower = -I₃
    # Lf B_upper = 0, Lg B_upper = I₃
    
    # CBF constraints become:
    # -α*B_upper ≤ u ≤ α*B_lower
    # which is: -α*(h_max - hw) ≤ u ≤ α*(hw - h_min)
    
    u_cbf_lower = -params.alpha_CBF * B_upper  # Lower bound from upper barrier
    u_cbf_upper = params.alpha_CBF * B_lower   # Upper bound from lower barrier
    
    # ============================================
    # Set up optimization problem
    # ============================================
    model = Model(HiGHS.Optimizer)
    set_silent(model)
    
    JuMP.@variable(model, u[1:3])
    
    # Compute gradV' * gx (only for [sigma; w] portion)
    gradV_gx = gradV_num' * gx_reduced
    
    # Objective function: minimize control effort weighted objective
    JuMP.@objective(model, Min, 
        gradV_gx * u + u' * R * u 
    )
    
    # ============================================
    # Constraints
    # ============================================
    
    # 1. Input torque constraints
    JuMP.@constraint(model, u .>= params.umin)
    JuMP.@constraint(model, u .<= params.umax)
    
    # 2. CBF constraints for RW momentum (element-wise)
    JuMP.@constraint(model, u .>= u_cbf_lower)
    JuMP.@constraint(model, u .<= u_cbf_upper)
    
    # Solve
    optimize!(model)
    
    # Check solution status
    status = termination_status(model)
    if status != MOI.OPTIMAL
        @warn "Optimization not optimal at t=$t. Status: $status"
    end
    
    # Extract solution
    u_val = value.(u)
    
    return u_val
end

In [ ]:
## Main simulation loop for GHJBCBF-Based Controller with RW Momentum Constraints
# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
x[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input
    u[:, i] = GHJBCBF_control_case1(t_grid[i], x[:, i], t_span[2], R, params)
    
    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end
end_time = time()
elapsed_time = end_time - start_time
println("Elapsed time: $elapsed_time seconds")
elapsed_time_GHJBCBF = elapsed_time

# Convert RPs to Euler angles for visualization
psi_GHJBCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the GHJBCBF-Based Controller
u_GHJBCBF = u
x_GHJBCBF = x
t_GHJBCBF = collect(t_grid)

# Cost calculation using trapezoidal rule
Cost_GHJBCBF = (
trapz(t_GHJBCBF, [x_GHJBCBF[1:6, i]' * Q * x_GHJBCBF[1:6, i] + 
                    u_GHJBCBF[:, i]' * R * u_GHJBCBF[:, i] for i in 1:length(t_grid)])
)
println("Cost J_GHJBCBF = $Cost_GHJBCBF")

# ============================================
# Check momentum wheel saturation (CORRECTED)
# ============================================
h_threshold = 0.02
hw_max_reached = any(x[7:9, :] .>= (params.hmax .- h_threshold))
hw_min_reached = any(x[7:9, :] .<= (params.hmin .+ h_threshold))

if hw_max_reached || hw_min_reached
    println("RW momentum saturation detected!")
    if hw_max_reached
        println("   - Maximum momentum limit approached")
    end
    if hw_min_reached
        println("   - Minimum momentum limit approached")
    end
else
    println("RW momentum stayed within safe bounds")
end

# ============================================
# Check control saturation (CORRECTED)
# ============================================
u_max_reached = any(u .>= (params.umax .- 0.001))
u_min_reached = any(u .<= (params.umin .+ 0.001))

if u_max_reached || u_min_reached
    println("Control torque saturation detected!")
else
    println("Control torque stayed within bounds")
end

In [ ]:
## Spacecraft Simulation Using Nonlinear Optimal Control (Midpoint/RK2 Dynamics)
# Problem parameters
xf = zeros(9)           # Desired final state [sigma; w; hw] all zeros
uf = zeros(3)           # Desired final control
dt = 0.1
n_steps = Int(t_span[2] / dt)

# Time grid (linear)
timepts = range(0, t_span[2], length=n_steps+1)

# Setup JuMP OCP
model = Model(Ipopt.Optimizer)
set_optimizer_attribute(model, "max_iter", 1000)
set_optimizer_attribute(model, "tol", 1e-10)
set_optimizer_attribute(model, "mu_strategy", "adaptive")
set_optimizer_attribute(model, "print_level", 1)

# Variables
@variable(model, X[1:9, 1:n_steps+1])      # [sigma; w; hw]
@variable(model, U[1:3, 1:n_steps+1])      # Control torques

# ============================================
# Helper: symbolic dynamics xdot = f(x) + g(x)*u for JuMP variables
# ============================================
J_inv_num = inv(params.J)

function ocp_xdot(sigma, w, hw, u)
    sigmac = [ 0.0       -sigma[3]  sigma[2]; 
               sigma[3]   0.0      -sigma[1]; 
              -sigma[2]   sigma[1]  0.0     ]
    
    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')
    
    wc = [ 0.0    -w[3]   w[2]; 
           w[3]    0.0   -w[1]; 
          -w[2]    w[1]   0.0 ]
    
    sigma_dot = M_sigma * w
    w_dot = J_inv_num * (-wc * (params.J * w + hw)) + J_inv_num * u
    hw_dot = -u
    
    return vcat(sigma_dot, w_dot, hw_dot)
end

# ============================================
# Explicit Midpoint (RK2) Dynamics Constraints
# x_{t+1} = x_t + dt * f(x_t + (dt/2)*f(x_t, u_t), u_t)
# Second-order accurate, minimal expression overhead
# ============================================
for t in 1:n_steps
    x_t = X[:, t]
    u_t = U[:, t]
    
    # Stage 1: evaluate at current state
    k1 = ocp_xdot(x_t[1:3], x_t[4:6], x_t[7:9], u_t)
    
    # Stage 2: evaluate at midpoint
    x_mid = x_t + (dt/2) * k1
    k2 = ocp_xdot(x_mid[1:3], x_mid[4:6], x_mid[7:9], u_t)
    
    # Midpoint update
    @constraint(model, X[:, t+1] .== x_t + dt * k2)
end

# Objective with trapezoidal integration
cost = 0
for t in 1:n_steps
    # Integrand at time t
    x_diff_t = X[1:6, t] - xf[1:6]
    integrand_t = x_diff_t' * Q * x_diff_t + U[:, t]' * R * U[:, t]
    
    # Integrand at time t+1
    x_diff_tnext = X[1:6, t+1] - xf[1:6]
    integrand_tnext = x_diff_tnext' * Q * x_diff_tnext + U[:, t+1]' * R * U[:, t+1]
    
    # Trapezoidal rule
    cost += (dt/2) * (integrand_t + integrand_tnext)
end

@objective(model, Min, cost)

# Initial condition
@constraint(model, X[:, 1] .== x0)

# ============================================
# State constraints
# ============================================
@constraint(model, [t=1:n_steps+1, i=1:3], params.hmin[i] <= X[i+6, t] <= params.hmax[i])

# ============================================
# Input constraints
# ============================================
@constraint(model, [t=1:n_steps+1, i=1:3], params.umin[i] <= U[i, t] <= params.umax[i])

# Solve
start_time = time()
optimize!(model)
end_time = time()

elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds ($(round(elapsed_time/60, digits=2)) minutes)")
elapsed_time_OCP = elapsed_time

# Extract results
x_OCP = value.(X)
u_OCP = value.(U)
t_OCP = collect(range(0, t_span[2], length=n_steps+1))
Cost_OCP = objective_value(model)
println("Cost_OCP: $Cost_OCP")
println("Optimization status: ", termination_status(model))

# Convert RPs to Euler angles
psi_OCP = hcat([rps_to_euler(x_OCP[1:3, i]) * (180 / π) for i in 1:size(x_OCP, 2)]...)'

# ============================================
# Check momentum wheel saturation
# ============================================
h_threshold = 0.01
hw_max_reached = any(x_OCP[7:9, :] .>= (params.hmax .- h_threshold))
hw_min_reached = any(x_OCP[7:9, :] .<= (params.hmin .+ h_threshold))

if hw_max_reached || hw_min_reached
    println("RW momentum saturation detected!")
    if hw_max_reached
        println("   - Maximum momentum limit approached")
    end
    if hw_min_reached
        println("   - Minimum momentum limit approached")
    end
else
    println("RW momentum stayed within safe bounds")
end

# ============================================
# Check control saturation
# ============================================
u_max_reached = any(u_OCP .>= (params.umax .- 0.001))
u_min_reached = any(u_OCP .<= (params.umin .+ 0.001))

if u_max_reached || u_min_reached
    println("Control torque saturation detected!")
else
    println("Control torque stayed within bounds")
end

In [ ]:
## OD-CLF-CBF-QP Controller with RW Momentum Constraints (Rodrigues Parameters)
function ODCLFCBF(t, x, R_gain, rho_0, p_rho, p_delta, params)
    # Extract state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Extract parameters
    J = params.J
    J_inv = inv(J)

    # ============================================
    # System Dynamics (matching spacecraft_plant)
    # ============================================
    sigmac = [0.0       -sigma[3]   sigma[2];
              sigma[3]   0.0       -sigma[1];
             -sigma[2]   sigma[1]   0.0     ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')

    wc = [0.0    -w[3]   w[2];
          w[3]    0.0   -w[1];
         -w[2]    w[1]   0.0 ]

    fx = vcat(
        M_sigma * w,
        J_inv * (-wc * (J * w + hw)),
        zeros(3)
    )

    gx = vcat(
        zeros(3, 3),
        J_inv,
        -I(3)
    )

    # ============================================
    # CLF Construction via Input-Output Linearization
    # ============================================
    # Output: y = sigma, relative degree r = 2
    # sigmadot = M(sigma) * w
    sigmadot = M_sigma * w

    # Ṁ(σ) = (1/2)(σ̇× + σ̇ σᵀ + σ σ̇ᵀ)
    sigmadotc = [0.0          -sigmadot[3]   sigmadot[2];
                 sigmadot[3]   0.0          -sigmadot[1];
                -sigmadot[2]   sigmadot[1]   0.0        ]

    Mdot_sigma = (1/2) * (sigmadotc + sigmadot * sigma' + sigma * sigmadot')

    # Second-order Lie derivatives
    # L_f²h = Ṁ(σ)ω + M(σ) J⁻¹ (−ω×(Jω + hw))
    # L_g L_f h = M(σ) J⁻¹
    L_f_2_h   = Mdot_sigma * w + M_sigma * J_inv * (-wc * (J * w + hw))
    L_g_L_f_h = M_sigma * J_inv

    # Feedforward term: u* = −(L_g L_f h)⁻¹ L_f² h
    L_mat = inv(L_g_L_f_h)
    ustar = -L_mat * L_f_2_h

    # Riccati equation matrices
    eta = vcat(sigma, sigmadot)

    F = [zeros(3, 3)            Matrix{Float64}(I, 3, 3);
         zeros(3, 3)            zeros(3, 3)             ]

    G_mat = [zeros(3, 3);
             Matrix{Float64}(I, 3, 3)]

    Q = Symmetric(Matrix{Float64}(I, 6, 6))

    R_mat = Symmetric(R_gain * (L_mat' * L_mat))

    # Solve CARE: Fᵀ P + P F + Q − P G R⁻¹ Gᵀ P = 0
    S = G_mat * inv(R_mat) * G_mat'
    S = Symmetric((S + S') / 2)    # Ensure exact symmetry for arec
    P, _, _ = arec(F, S, Q)

    # CLF value and Lie derivatives
    V_val    = dot(eta, P * eta)
    L_fbar_V = dot(eta, (F' * P + P * F) * eta)
    L_gbar_V = 2.0 * eta' * P * G_mat            # 1×3 row vector
    W_val    = dot(eta, (Q + P * G_mat * inv(R_mat) * G_mat' * P) * eta)

    # CLF constraint coefficient: L_ḡ V · L̄(·)
    coeff_clf = vec(L_gbar_V * L_g_L_f_h)         # 3-element vector

    # Objective weighting
    H = Matrix{Float64}(I, 3, 3)
    Hbar = L_g_L_f_h' * H * L_g_L_f_h

    # ============================================
    # Control Barrier Function (CBF) Constraints
    # ============================================
    # CBF for RW momentum: h_min ≤ hw ≤ h_max
    #   B_lower(x) = hw - h_min ≥ 0
    #   B_upper(x) = h_max - hw ≥ 0
    # Lf B = 0, Lg B_lower = -I₃, Lg B_upper = I₃
    # CBF constraints: -α(h_max - hw) ≤ u ≤ α(hw - h_min)

    B_lower = hw - params.hmin
    B_upper = params.hmax - hw

    u_cbf_lower = -params.alpha_CBF * B_upper
    u_cbf_upper =  params.alpha_CBF * B_lower

    # ============================================
    # Optimal-Decay CLF-CBF-QP
    # ============================================
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, u[1:3])
    @variable(model, rho >= 0)
    @variable(model, delta)

    # Objective: ‖L̄(u − u*)‖²_H  +  p_ρ (1−ρ)²  +  p_δ δ²
    @objective(model, Min,
        sum(Hbar[i, j] * (u[i] - ustar[i]) * (u[j] - ustar[j])
            for i in 1:3, j in 1:3) +
        p_rho * (1.0 - rho)^2 +
        p_delta * delta^2
    )

    # CLF constraint: L_f̄ V + L_ḡ V · L̄(u − u*) ≤ −ρ W + δ
    @constraint(model,
        L_fbar_V + sum(coeff_clf[i] * (u[i] - ustar[i]) for i in 1:3) <=
        -rho * W_val + delta
    )

    # CBF constraints for RW momentum (element-wise)
    @constraint(model, u .>= u_cbf_lower)
    @constraint(model, u .<= u_cbf_upper)

    # Input torque constraints
    @constraint(model, u .>= params.umin)
    @constraint(model, u .<= params.umax)

    # ============================================
    # Solve
    # ============================================
    optimize!(model)

    status = termination_status(model)
    if status != MOI.OPTIMAL
        @warn "Optimization not optimal at t=$t. Status: $status"
    end

    # Extract solution
    u_val     = value.(u)
    rho_val   = value(rho)
    delta_val = value(delta)

    return u_val, rho_val, delta_val
end

In [ ]:
## Main simulation - OD-CLF-CBF-QP Controller with RW Momentum Constraints
rho_0 = 1.0
p_rho = 0.1
p_delta = 100.0
R_gain = 10.0

# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
rho_hist = zeros(n_t_grid)
delta_hist = zeros(n_t_grid)
x[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input, decay weight, and slack variable
    u_val, rho_val, delta_val = ODCLFCBF(t_grid[i], x[:, i], R_gain, rho_0, p_rho, p_delta, params)
    u[:, i] = u_val
    rho_hist[i] = rho_val
    delta_hist[i] = delta_val

    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end
end_time = time()
elapsed_time = end_time - start_time
println("Elapsed time: $elapsed_time seconds")
elapsed_time_ODCLFCBF = elapsed_time

# Convert RPs to Euler angles for visualization
psi_ODCLFCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the OD-CLF-CBF-QP Controller
u_ODCLFCBF = u
x_ODCLFCBF = x
t_ODCLFCBF = collect(t_grid)
rho_ODCLFCBF = rho_hist
delta_ODCLFCBF = delta_hist

# Cost calculation using trapezoidal rule
Cost_ODCLFCBF = (
    trapz(t_ODCLFCBF, [x_ODCLFCBF[1:6, i]' * Q * x_ODCLFCBF[1:6, i] + 
                        u_ODCLFCBF[:, i]' * R * u_ODCLFCBF[:, i] for i in 1:length(t_grid)])
)
println("Cost J_ODCLFCBF = $Cost_ODCLFCBF")

# ============================================
# Check asymptotic stability via slack variable
# ============================================
if abs(delta_hist[end]) < 1e-6
    println("δ → 0: asymptotic stability assured")
else
    println("δ = $(delta_hist[end]) at final time (not yet converged)")
end

# ============================================
# Check momentum wheel saturation
# ============================================
h_threshold = 0.02
hw_max_reached = any(x[7:9, :] .>= (params.hmax .- h_threshold))
hw_min_reached = any(x[7:9, :] .<= (params.hmin .+ h_threshold))

if hw_max_reached || hw_min_reached
    println("RW momentum saturation detected!")
    if hw_max_reached
        println("   - Maximum momentum limit approached")
    end
    if hw_min_reached
        println("   - Minimum momentum limit approached")
    end
else
    println("RW momentum stayed within safe bounds")
end

# ============================================
# Check control saturation
# ============================================
u_max_reached = any(u .>= (params.umax .- 0.001))
u_min_reached = any(u .<= (params.umin .+ 0.001))

if u_max_reached || u_min_reached
    println("Control torque saturation detected!")
else
    println("Control torque stayed within bounds")
end

In [ ]:
## RES-CLF-CBF-QP Controller with RW Momentum Constraints (Rodrigues Parameters)
# Based on W(η) = (1/ε)(λ_min(Q)/λ_max(P)) V(η)
function RESCLFCBF(t, x, R_gain, epsilon, p_delta, params)
    # Extract state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Extract parameters
    J = params.J
    J_inv = inv(J)

    # ============================================
    # System Dynamics (matching spacecraft_plant)
    # ============================================
    sigmac = [0.0       -sigma[3]   sigma[2];
              sigma[3]   0.0       -sigma[1];
             -sigma[2]   sigma[1]   0.0     ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')

    wc = [0.0    -w[3]   w[2];
          w[3]    0.0   -w[1];
         -w[2]    w[1]   0.0 ]

    fx = vcat(
        M_sigma * w,
        J_inv * (-wc * (J * w + hw)),
        zeros(3)
    )

    gx = vcat(
        zeros(3, 3),
        J_inv,
        -I(3)
    )

    # ============================================
    # CLF Construction via Input-Output Linearization
    # ============================================
    # Output: y = sigma, relative degree r = 2
    sigmadot = M_sigma * w

    # Ṁ(σ) = (1/2)(σ̇× + σ̇ σᵀ + σ σ̇ᵀ)
    sigmadotc = [0.0          -sigmadot[3]   sigmadot[2];
                 sigmadot[3]   0.0          -sigmadot[1];
                -sigmadot[2]   sigmadot[1]   0.0        ]

    Mdot_sigma = (1/2) * (sigmadotc + sigmadot * sigma' + sigma * sigmadot')

    # Second-order Lie derivatives
    L_f_2_h   = Mdot_sigma * w + M_sigma * J_inv * (-wc * (J * w + hw))
    L_g_L_f_h = M_sigma * J_inv

    # Feedforward term: u* = −(L_g L_f h)⁻¹ L_f² h
    L_mat = inv(L_g_L_f_h)
    ustar = -L_mat * L_f_2_h

    # Riccati equation matrices
    eta = vcat(sigma, sigmadot)

    F = [zeros(3, 3)            Matrix{Float64}(I, 3, 3);
         zeros(3, 3)            zeros(3, 3)             ]

    G_mat = [zeros(3, 3);
             Matrix{Float64}(I, 3, 3)]

    Q = Symmetric(Matrix{Float64}(I, 6, 6))

    R_mat = Symmetric(R_gain * (L_mat' * L_mat))

    # Solve CARE: Fᵀ P + P F + Q − P G R⁻¹ Gᵀ P = 0
    S = G_mat * inv(R_mat) * G_mat'
    S = Symmetric((S + S') / 2)
    P, _, _ = arec(F, S, Q)

    # CLF value and Lie derivatives
    V_val    = dot(eta, P * eta)
    L_fbar_V = dot(eta, (F' * P + P * F) * eta)
    L_gbar_V = 2.0 * eta' * P * G_mat            # 1×3 row vector

    # ============================================
    # RES-CLF decay rate (Eq. 14, Remark 3)
    # W(η) = (1/ε)(λ_min(Q)/λ_max(P)) V(η)
    # ============================================
    lambda_min_Q = minimum(eigvals(Matrix(Q)))
    lambda_max_P = maximum(eigvals(P))
    W_val = (1.0 / epsilon) * (lambda_min_Q / lambda_max_P) * V_val

    # CLF constraint coefficient: L_ḡ V · L_g L_f h
    coeff_clf = vec(L_gbar_V * L_g_L_f_h)         # 3-element vector

    # Objective weighting
    H = Matrix{Float64}(I, 3, 3)
    Hbar = L_g_L_f_h' * H * L_g_L_f_h

    # ============================================
    # Control Barrier Function (CBF) Constraints
    # ============================================
    B_lower = hw - params.hmin
    B_upper = params.hmax - hw

    u_cbf_lower = -params.alpha_CBF * B_upper
    u_cbf_upper =  params.alpha_CBF * B_lower

    # ============================================
    # RES-CLF-CBF-QP
    # ============================================
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, u[1:3])
    @variable(model, delta)

    # Objective: ‖L̄(u − u*)‖²_H  +  p_δ δ²
    @objective(model, Min,
        sum(Hbar[i, j] * (u[i] - ustar[i]) * (u[j] - ustar[j])
            for i in 1:3, j in 1:3) +
        p_delta * delta^2
    )

    # CLF constraint (fixed decay, no ρ): L_f̄ V + L_ḡ V · L̄(u − u*) ≤ −W + δ
    @constraint(model,
        L_fbar_V + sum(coeff_clf[i] * (u[i] - ustar[i]) for i in 1:3) <=
        -W_val + delta
    )

    # CBF constraints for RW momentum (element-wise)
    @constraint(model, u .>= u_cbf_lower)
    @constraint(model, u .<= u_cbf_upper)

    # Input torque constraints
    @constraint(model, u .>= params.umin)
    @constraint(model, u .<= params.umax)

    # ============================================
    # Solve
    # ============================================
    optimize!(model)

    status = termination_status(model)
    if status != MOI.OPTIMAL
        @warn "Optimization not optimal at t=$t. Status: $status"
    end

    # Extract solution
    u_val     = value.(u)
    delta_val = value(delta)

    return u_val, delta_val
end

In [ ]:
## Main simulation - RES-CLF-CBF-QP Controller
epsilon = 0.02
p_delta = 100.0
R_gain = 10.0

# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
delta_hist = zeros(n_t_grid)
x[:, 1] = x0

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input and slack variable (no rho)
    u_val, delta_val = RESCLFCBF(t_grid[i], x[:, i], R_gain, epsilon, p_delta, params)
    u[:, i] = u_val
    delta_hist[i] = delta_val

    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end
end_time = time()
elapsed_time = end_time - start_time
println("Elapsed time: $elapsed_time seconds")
elapsed_time_RESCLFCBF = elapsed_time

# Convert RPs to Euler angles for visualization
psi_RESCLFCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the RES-CLF-CBF-QP Controller
u_RESCLFCBF = u
x_RESCLFCBF = x
t_RESCLFCBF = collect(t_grid)
delta_RESCLFCBF = delta_hist

# Cost calculation using trapezoidal rule
Cost_RESCLFCBF = (
    trapz(t_RESCLFCBF, [x_RESCLFCBF[1:6, i]' * Q * x_RESCLFCBF[1:6, i] + 
                        u_RESCLFCBF[:, i]' * R * u_RESCLFCBF[:, i] for i in 1:length(t_grid)])
)
println("Cost J_RESCLFCBF = $Cost_RESCLFCBF")

# ============================================
# Check asymptotic stability via slack variable
# ============================================
if abs(delta_hist[end]) < 1e-6
    println("δ → 0: asymptotic stability assured")
else
    println("δ = $(delta_hist[end]) at final time (not yet converged)")
end

# ============================================
# Check momentum wheel saturation
# ============================================
h_threshold = 0.02
hw_max_reached = any(x[7:9, :] .>= (params.hmax .- h_threshold))
hw_min_reached = any(x[7:9, :] .<= (params.hmin .+ h_threshold))

if hw_max_reached || hw_min_reached
    println("RW momentum saturation detected!")
    if hw_max_reached
        println("   - Maximum momentum limit approached")
    end
    if hw_min_reached
        println("   - Minimum momentum limit approached")
    end
else
    println("RW momentum stayed within safe bounds")
end

# ============================================
# Check control saturation
# ============================================
u_max_reached = any(u .>= (params.umax .- 0.001))
u_min_reached = any(u .<= (params.umin .+ 0.001))

if u_max_reached || u_min_reached
    println("Control torque saturation detected!")
else
    println("Control torque stayed within bounds")
end

In [ ]:
## ==========================================================================
## Plot Trajectories — Spacecraft
## ==========================================================================
# Requires: 
#           x_RESCLFCBF, u_RESCLFCBF, t_RESCLFCBF                (RES-CLF-CBF-QP)
#           x_ODCLFCBF, u_ODCLFCBF, t_ODCLFCBF                   (OD-CLF-CBF-QP)
#           x_OCP, u_OCP, t_OCP                                  (Optimal Control)
#           x_GHJBCBF, u_GHJBCBF, t_GHJBCBF                      (GHJB-CBF-QP)

mkpath("Example_2_Figures")

selected_controllers = ["1", "2", "3", "4"]
input_list = ["1", "2", "3", "4"]
# Flexible data storage
controllers_data = Dict(
    "1" => Dict("label" => "RES-CLF-CBF-QP", "color" => "tab:green", 
                "x" => x_RESCLFCBF, "u" => u_RESCLFCBF, "t" => t_RESCLFCBF),
    "2" => Dict("label" => "OD-CLF-CBF-QP", "color" => "tab:orange", 
                "x" => x_ODCLFCBF, "u" => u_ODCLFCBF, "t" => t_ODCLFCBF),
    "3" => Dict("label" => "Open-Loop Optimal (OCP)", "color" => "tab:blue", 
                "x" => x_OCP, "u" => u_OCP, "t" => t_OCP),
    "4" => Dict("label" => "GHJB-CBF-QP", "color" => "black", 
                "x" => x_GHJBCBF, "u" => u_GHJBCBF, "t" => t_GHJBCBF)
)
for i in 1:3
    # Create subplots
    fig, (ax1, ax2, ax3, ax4) = subplots(4, figsize=(3.41, 4.0), sharex=true)
    fig.set_constrained_layout(true)
    
    # Plot for each selected controller
    for controller in selected_controllers
        if controller in input_list
            data = controllers_data[controller]
            ax1.plot(data["t"], data["x"][i, :], linestyle="solid", color=data["color"], 
                    zorder=3, label=data["label"])
            ax2.plot(data["t"], data["x"][i+3, :], linestyle="solid", color=data["color"], 
                    zorder=3)
            ax3.plot(data["t"], data["x"][i+6, :], linestyle="solid", color=data["color"], 
                    zorder=3)
            ax4.plot(data["t"], data["u"][i, :], linestyle="solid", color=data["color"], 
                    zorder=3)
        end
    end
    
    # Add reference lines and labels
    ax1.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax1.set_ylabel(L"\sigma_{%$(i)}")
    
    ax2.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax2.set_ylabel(L"\omega_{%$(i)} \text{ (rad/s)}")
    
    # ============================================
    # RW momentum constraints (CORRECTED)
    # ============================================
    ax3.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax3.hlines(y=params.hmax[i], xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2, label="Input/State Constraints")
    ax3.hlines(y=params.hmin[i], xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
    ax3.set_ylabel(L"h_{w%$(i)} \text{ (N m s)}")
    
    # ============================================
    # Control torque constraints (CORRECTED)
    # ============================================
    ax4.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax4.hlines(y=params.umax[i], xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
    ax4.hlines(y=params.umin[i], xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
    ax4.set_ylabel(L"u_{%$(i)} \text{ (N m)}")
    # ============================================
    # Fill to padded plot edges
    # ============================================
    yl3 = ax3.get_ylim()
    pad3 = 0.05 * (yl3[2] - yl3[1])  # 5% padding — adjust to taste
    new_yl3 = (min(yl3[1], params.hmin[i]) - pad3, max(yl3[2], params.hmax[i]) + pad3)
    ax3.fill_between(t_GHJBCBF, params.hmax[i], new_yl3[2], alpha=0.2, color="red")
    ax3.fill_between(t_GHJBCBF, new_yl3[1], params.hmin[i], alpha=0.2, color="red")
    ax3.set_ylim(new_yl3)
    yl4 = ax4.get_ylim()
    pad4 = 0.05 * (yl4[2] - yl4[1])  # 5% padding — adjust to taste
    new_yl4 = (min(yl4[1], params.umin[i]) - pad4, max(yl4[2], params.umax[i]) + pad4)
    ax4.fill_between(t_GHJBCBF, params.umax[i], new_yl4[2], alpha=0.2, color="red")
    ax4.fill_between(t_GHJBCBF, new_yl4[1], params.umin[i], alpha=0.2, color="red")
    ax4.set_ylim(new_yl4)
    # ============================================
    # Autoscale and add grids
    for ax in [ax1, ax2, ax3, ax4]
        ax.autoscale(enable=true, axis="x", tight=true)
        ax.grid(true, zorder=0, color="0.95", linestyle="dotted", which="both", axis="both")
    end
    
    # Set x-axis label
    ax4.set_xlabel("Time (s)")
    
    # Get legend handles and labels
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles3, labels3 = ax3.get_legend_handles_labels()
    handles = vcat(handles1, handles3)
    labels = vcat(labels1, labels3)
    
    if length(handles) > 0  # Only show legend if there are selected controllers
        fig.legend(handles, labels, loc="outside upper center", ncol=2, 
                  framealpha=0, edgecolor="none", facecolor="white").set_zorder(2)
    end
    
    # Show and save the figure
    # show()
    savefig("Example_2_Figures/trajectories_spacecraft_case1_$(i).pdf", dpi=300, bbox_inches="tight", pad_inches=0.0)
    println("Saved: Example_2_Figures/trajectories_spacecraft_case1_$(i).pdf")
end

## Case 2: Forbidden pointing constraint

In [ ]:
## GHJBCBF Controller with Forbidden Pointing Constraints (Case 2)
function GHJBCBF_control_case2(t, x, t_span_end, R, params)
    # Define state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Compute V
    V_sym = results.V
    vars = DynamicPolynomials.variables(V_sym)

    # Compute the gradient of V with respect to state variables
    gradV_sym = [DynamicPolynomials.differentiate(V_sym, v) for v in vars]
    gradV_num = DynamicPolynomials.subs(gradV_sym, vars => x[1:6])
    gradV_num = first.(DynamicPolynomials.coefficients.(gradV_num))

    # System matrices (same notation as spacecraft_plant)
    J = params.J
    J_inv = inv(J)

    sigmac = [  0.0       -sigma[3]   sigma[2]; 
                sigma[3]   0.0       -sigma[1]; 
               -sigma[2]   sigma[1]   0.0      ]

    wc = [  0.0   -w[3]   w[2]; 
            w[3]   0.0   -w[1]; 
           -w[2]   w[1]   0.0  ]

    gx = vcat(zeros(3, 3), J_inv)

    # Drift part of ω̇ (unforced): J⁻¹(-ω×(Jω + hw))
    wdot_drift = J_inv * (-wc * (J * w + hw))

    # ============================================
    # Rotation matrix R(σ): inertial → body
    # ============================================
    R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)

    # HOCBF parameters
    alpha1_hocbf = params.alpha1_HOCBF
    alpha2_hocbf = params.alpha2_HOCBF

    # Set up optimization problem
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    JuMP.@variable(model, u[1:3])

    # Compute gradV_num' * gx
    gradV_num_gx = gradV_num' * gx

    # Objective function
    JuMP.@objective(model, Min, 
        gradV_num_gx * u + u' * R * u
    )

    # ============================================
    # Input constraints
    # ============================================
    JuMP.@constraint(model, u .>= params.umin)
    JuMP.@constraint(model, u .<= params.umax)

    # ============================================
    # HOCBF pointing constraints
    # ============================================
    for i in eachindex(params.b_sensors)
        b_i = params.b_sensors[i]
        theta_i = params.theta_fov[i]

        for j in eachindex(params.n_bright)
            n_j = params.n_bright[j]

            # Bright object direction in body frame
            r_j = R_sigma * n_j

            # c = rⱼ × bᵢ
            c = cross(r_j, b_i)

            # ψ₀ = cos(θᵢ) - bᵢᵀ rⱼ
            b_val = cos(theta_i) - dot(b_i, r_j)

            # L_F b = cᵀ ω
            Lf_b = dot(c, w)

            # ċ = ṙⱼ × bᵢ = (-wc * rⱼ) × bᵢ = bᵢ × (ω × rⱼ)
            c_dot = cross(b_i, wc * r_j)

            # L²_F b = ċᵀω + cᵀ(J⁻¹(-ω×(Jω + hw)))
            Lf2_b = dot(c_dot, w) + dot(c, wdot_drift)

            # L_G L_F b = cᵀ J⁻¹   (3-vector, coefficients of u)
            LgLf_b = c' * J_inv

            # HOCBF constraint: L²b + LgLf·u + (α₁+α₂)Lfb + α₁α₂·b ≥ 0
            JuMP.@constraint(model, 
                Lf2_b + LgLf_b * u + (alpha1_hocbf + alpha2_hocbf) * Lf_b + 
                alpha1_hocbf * alpha2_hocbf * b_val >= 0
            )
        end
    end

    # Solve
    optimize!(model)
    u_val = value.(u)

    return u_val
end

In [ ]:
# Function to compute pointing constraint value
function compute_pointing_constraint(sigma, b_i, n_j, theta_fov)
    sigmac = [  0.0       -sigma[3]   sigma[2]; 
                sigma[3]   0.0       -sigma[1]; 
               -sigma[2]   sigma[1]   0.0      ]
    
    R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)
    
    # r = R(σ) nⱼ (bright object direction in body frame)
    r = R_sigma * n_j
    
    # Constraint: b = cos(θ) - bᵢᵀr ≥ 0
    b_val = cos(theta_fov) - dot(b_i, r)
    
    # Actual angle between sensor and bright object
    cos_angle = dot(b_i, r)
    angle_rad = acos(clamp(cos_angle, -1.0, 1.0))
    angle_deg = rad2deg(angle_rad)
    
    return b_val, angle_deg
end

In [ ]:
## Main simulation for GHJBCBF with Pointing Constraints (Case 2)
# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
x[:, 1] = x0

# Storage for pointing constraint data
n_sensors = length(params.b_sensors)
n_bright_objs = length(params.n_bright)
n_constraints = n_sensors * n_bright_objs
b_vals = zeros(n_constraints, n_t_grid)
angles_deg = zeros(n_constraints, n_t_grid)

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input
    u[:, i] = GHJBCBF_control_case2(t_grid[i], x[:, i], t_span[2], R, params)
    
    # Compute pointing constraint values at current state
    sigma_current = x[1:3, i]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, angle_deg = compute_pointing_constraint(
                sigma_current,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals[idx, i] = b_val
            angles_deg[idx, i] = angle_deg
            idx += 1
        end
    end
    
    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end

# Compute final time step pointing constraints
sigma_final = x[1:3, end]
idx = 1
for s in eachindex(params.b_sensors)
    for j in eachindex(params.n_bright)
        b_val, angle_deg = compute_pointing_constraint(
            sigma_final,
            params.b_sensors[s],
            params.n_bright[j],
            params.theta_fov[s]
        )
        b_vals[idx, end] = b_val
        angles_deg[idx, end] = angle_deg
        idx += 1
    end
end

end_time = time()
elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds")
elapsed_time_SOSCBF = elapsed_time

# ============================================
# CORRECTED: rps_to_euler instead of mrps_to_euler
# ============================================
psi_GHJBCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the SOSCBF-Based Controller
u_GHJBCBF = u
x_GHJBCBF = x
t_GHJBCBF = collect(t_grid)
b_vals_GHJBCBF = b_vals
angles_deg_GHJBCBF = angles_deg

# Cost calculation
Cost_GHJBCBF = (
    trapz(t_GHJBCBF, [x_GHJBCBF[1:6, i]' * Q * x_GHJBCBF[1:6, i] + 
                       u_GHJBCBF[:, i]' * R * u_GHJBCBF[:, i] for i in 1:length(t_grid)])
)

println("Cost J_GHJBCBF = $Cost_GHJBCBF")

# Check pointing constraint satisfaction
min_b_ghjbcbf = minimum(b_vals_GHJBCBF)
println("GHJBCBF minimum pointing constraint value: $(round(min_b_ghjbcbf, digits=6))")

violation_tol = 1e-3
if min_b_ghjbcbf >= - violation_tol
    println("GHJBCBF pointing constraint satisfied throughout trajectory")
else
    println("GHJBCBF pointing constraint VIOLATED! (min b = $(round(min_b_ghjbcbf, digits=6)))")
end

In [ ]:
## Spacecraft Simulation Using Nonlinear Optimal Control with Pointing Constraints (RK2 Dynamics)
# Problem parameters
xf = zeros(9)           # Desired final state [sigma; w; hw] all zeros
uf = zeros(3)           # Desired final control
dt = 0.1
n_steps = Int(t_span[2] / dt)

# Time grid (linear)
timepts = range(0, t_span[2], length=n_steps+1)

# Setup JuMP OCP
model = Model(Ipopt.Optimizer)
set_optimizer_attribute(model, "max_iter", 1000)
set_optimizer_attribute(model, "tol", 1e-10)
set_optimizer_attribute(model, "mu_strategy", "adaptive")
set_optimizer_attribute(model, "print_level", 1)

# Variables
@variable(model, X[1:9, 1:n_steps+1])      # [sigma; w; hw]
@variable(model, U[1:3, 1:n_steps+1])      # Control torques

# ============================================
# Helper: symbolic dynamics xdot = f(x) + g(x)*u for JuMP variables
# ============================================
J_inv_num = inv(params.J)

function ocp_xdot(sigma, w, hw, u)
    sigmac = [ 0.0       -sigma[3]  sigma[2]; 
               sigma[3]   0.0      -sigma[1]; 
              -sigma[2]   sigma[1]  0.0     ]
    
    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')
    
    wc = [ 0.0    -w[3]   w[2]; 
           w[3]    0.0   -w[1]; 
          -w[2]    w[1]   0.0 ]
    
    sigma_dot = M_sigma * w
    w_dot = J_inv_num * (-wc * (params.J * w + hw)) + J_inv_num * u
    hw_dot = -u
    
    return vcat(sigma_dot, w_dot, hw_dot)
end

# ============================================
# Explicit Midpoint (RK2) Dynamics Constraints
# ============================================
for t in 1:n_steps
    x_t = X[:, t]
    u_t = U[:, t]
    
    # Stage 1: evaluate at current state
    k1 = ocp_xdot(x_t[1:3], x_t[4:6], x_t[7:9], u_t)
    
    # Stage 2: evaluate at midpoint
    x_mid = x_t + (dt/2) * k1
    k2 = ocp_xdot(x_mid[1:3], x_mid[4:6], x_mid[7:9], u_t)
    
    # Midpoint update
    @constraint(model, X[:, t+1] .== x_t + dt * k2)
end

# Objective with trapezoidal integration
cost = 0
for t in 1:n_steps
    # Integrand at time t
    x_diff_t = X[1:6, t] - xf[1:6]
    integrand_t = x_diff_t' * Q * x_diff_t + U[:, t]' * R * U[:, t]
    
    # Integrand at time t+1
    x_diff_tnext = X[1:6, t+1] - xf[1:6]
    integrand_tnext = x_diff_tnext' * Q * x_diff_tnext + U[:, t+1]' * R * U[:, t+1]
    
    # Trapezoidal rule
    cost += (dt/2) * (integrand_t + integrand_tnext)
end

@objective(model, Min, cost)

# Initial condition
@constraint(model, X[:, 1] .== x0)

# Input constraints
@constraint(model, [t=1:n_steps+1, i=1:3], params.umin[i] <= U[i, t] <= params.umax[i])

# ============================================
# Pointing Constraints using R(σ) matrix form
# ============================================
# R(σ) = (1/(1+σᵀσ)) * [(1-σᵀσ)I + 2σσᵀ - 2σ̃]
# b(σ) = cos(θ) - bᵢᵀ R(σ) nⱼ ≥ 0
for t in 1:n_steps+1
    sigma = X[1:3, t]
    
    sigmac = [ 0.0       -sigma[3]  sigma[2]; 
               sigma[3]   0.0      -sigma[1]; 
              -sigma[2]   sigma[1]  0.0     ]
    
    R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)
    
    for s in eachindex(params.b_sensors)
        b_i = params.b_sensors[s]
        theta_i = params.theta_fov[s]
        
        for j in eachindex(params.n_bright)
            n_j = params.n_bright[j]
            r_j = R_sigma * n_j
            @constraint(model, cos(theta_i) - b_i' * r_j >= 0)
        end
    end
end

# Solve
start_time = time()
optimize!(model)
end_time = time()

elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds ($(round(elapsed_time/60, digits=2)) minutes)")
elapsed_time_OCP = elapsed_time

# Extract results
x_OCP = value.(X)
u_OCP = value.(U)
t_OCP = collect(range(0, t_span[2], length=n_steps+1))
Cost_OCP = objective_value(model)
println("Cost_OCP: $Cost_OCP")
println("Optimization status: ", termination_status(model))

# Convert RPs to Euler angles
psi_OCP = hcat([rps_to_euler(x_OCP[1:3, i]) * (180 / π) for i in 1:size(x_OCP, 2)]...)'

# Check control saturation
u_threshold = 0.001
u_max_reached = any(u_OCP .>= (params.umax .- u_threshold))
u_min_reached = any(u_OCP .<= (params.umin .+ u_threshold))

if u_max_reached || u_min_reached
    println("Control torque saturation detected!")
else
    println("Control torque stayed within bounds")
end

# ============================================
# Check pointing constraint satisfaction
# ============================================
n_sensors = length(params.b_sensors)
n_bright_objs = length(params.n_bright)
n_constraints = n_sensors * n_bright_objs
b_vals_OCP = zeros(n_constraints, n_steps+1)
angles_deg_OCP = zeros(n_constraints, n_steps+1)

for t in 1:n_steps+1
    sigma_t = x_OCP[1:3, t]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, angle_deg = compute_pointing_constraint(
                sigma_t,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals_OCP[idx, t] = b_val
            angles_deg_OCP[idx, t] = angle_deg
            idx += 1
        end
    end
end

min_b_OCP = minimum(b_vals_OCP)
println("OCP minimum pointing constraint value: $(round(min_b_OCP, digits=6))")

violation_tol = 1e-3
if min_b_OCP >= -violation_tol
    println("OCP pointing constraint satisfied throughout trajectory")
else
    println("OCP pointing constraint VIOLATED! (min b = $(round(min_b_OCP, digits=6)))")
end

In [ ]:
## OD-CLF-CBF-QP Controller - Case 2: Pointing Constraints
function ODCLFCBF_control_pointing(t, x, R_gain, p_rho, p_delta, params)
    # Extract state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Extract parameters
    J = params.J
    J_inv = inv(J)

    # ============================================
    # System Dynamics (matching spacecraft_plant)
    # ============================================
    sigmac = [0.0       -sigma[3]   sigma[2];
              sigma[3]   0.0       -sigma[1];
             -sigma[2]   sigma[1]   0.0     ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')

    wc = [0.0    -w[3]   w[2];
          w[3]    0.0   -w[1];
         -w[2]    w[1]   0.0 ]

    # Drift part of ω̇ (unforced): J⁻¹(-ω×(Jω + hw))
    wdot_drift = J_inv * (-wc * (J * w + hw))

    # ============================================
    # CLF Construction via Input-Output Linearization
    # ============================================
    # Output: y = sigma, relative degree r = 2
    sigmadot = M_sigma * w

    # Ṁ(σ) = (1/2)(σ̇× + σ̇ σᵀ + σ σ̇ᵀ)
    sigmadotc = [0.0          -sigmadot[3]   sigmadot[2];
                 sigmadot[3]   0.0          -sigmadot[1];
                -sigmadot[2]   sigmadot[1]   0.0        ]

    Mdot_sigma = (1/2) * (sigmadotc + sigmadot * sigma' + sigma * sigmadot')

    # Second-order Lie derivatives
    L_f_2_h   = Mdot_sigma * w + M_sigma * wdot_drift
    L_g_L_f_h = M_sigma * J_inv

    # Feedforward term: u* = −(L_g L_f h)⁻¹ L_f² h
    L_mat = inv(L_g_L_f_h)
    ustar = -L_mat * L_f_2_h

    # Riccati equation matrices
    eta = vcat(sigma, sigmadot)

    F = [zeros(3, 3)            Matrix{Float64}(I, 3, 3);
         zeros(3, 3)            zeros(3, 3)             ]

    G_mat = [zeros(3, 3);
             Matrix{Float64}(I, 3, 3)]

    Q = Symmetric(Matrix{Float64}(I, 6, 6))

    R_mat = Symmetric(R_gain * (L_mat' * L_mat))

    # Solve CARE: Fᵀ P + P F + Q − P G R⁻¹ Gᵀ P = 0
    S = G_mat * inv(R_mat) * G_mat'
    S = Symmetric((S + S') / 2)
    P, _, _ = arec(F, S, Q)

    # CLF value and Lie derivatives
    V_val    = dot(eta, P * eta)
    L_fbar_V = dot(eta, (F' * P + P * F) * eta)
    L_gbar_V = 2.0 * eta' * P * G_mat            # 1×3 row vector
    W_val    = dot(eta, (Q + P * G_mat * inv(R_mat) * G_mat' * P) * eta)

    # CLF constraint coefficient: L_ḡ V · L_g L_f h
    coeff_clf = vec(L_gbar_V * L_g_L_f_h)         # 3-element vector

    # Objective weighting
    H = Matrix{Float64}(I, 3, 3)
    Hbar = L_g_L_f_h' * H * L_g_L_f_h

    # ============================================
    # Rotation matrix R(σ): inertial → body
    # ============================================
    R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)

    # HOCBF parameters
    alpha1_hocbf = params.alpha1_HOCBF
    alpha2_hocbf = params.alpha2_HOCBF

    # ============================================
    # Optimal-Decay CLF-QP with HOCBF Pointing
    # ============================================
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, u[1:3])
    @variable(model, rho >= 0)
    @variable(model, delta)

    # Objective: ‖L̄(u − u*)‖²_H  +  p_ρ (1−ρ)²  +  p_δ δ²
    @objective(model, Min,
        sum(Hbar[i, j] * (u[i] - ustar[i]) * (u[j] - ustar[j])
            for i in 1:3, j in 1:3) +
        p_rho * (1.0 - rho)^2 +
        p_delta * delta^2
    )

    # CLF constraint: L_f̄ V + L_ḡ V · L̄(u − u*) ≤ −ρ W + δ
    @constraint(model,
        L_fbar_V + sum(coeff_clf[i] * (u[i] - ustar[i]) for i in 1:3) <=
        -rho * W_val + delta
    )

    # ============================================
    # Input constraints
    # ============================================
    @constraint(model, u .>= params.umin)
    @constraint(model, u .<= params.umax)

    # ============================================
    # HOCBF pointing constraints
    # ============================================
    for i in eachindex(params.b_sensors)
        b_i = params.b_sensors[i]
        theta_i = params.theta_fov[i]

        for j in eachindex(params.n_bright)
            n_j = params.n_bright[j]

            # Bright object direction in body frame
            r_j = R_sigma * n_j

            # c = rⱼ × bᵢ
            c = cross(r_j, b_i)

            # ψ₀ = cos(θᵢ) - bᵢᵀ rⱼ
            b_val = cos(theta_i) - dot(b_i, r_j)

            # L_F b = cᵀ ω
            Lf_b = dot(c, w)

            # ċ = ṙⱼ × bᵢ = (-wc * rⱼ) × bᵢ = bᵢ × (ω × rⱼ)
            c_dot = cross(b_i, wc * r_j)

            # L²_F b = ċᵀω + cᵀ(J⁻¹(-ω×(Jω + hw)))
            Lf2_b = dot(c_dot, w) + dot(c, wdot_drift)

            # L_G L_F b = cᵀ J⁻¹   (3-vector, coefficients of u)
            LgLf_b = c' * J_inv

            # HOCBF constraint: L²b + LgLf·u + (α₁+α₂)Lfb + α₁α₂·b ≥ 0
            @constraint(model,
                Lf2_b + LgLf_b * u + (alpha1_hocbf + alpha2_hocbf) * Lf_b +
                alpha1_hocbf * alpha2_hocbf * b_val >= 0
            )
        end
    end

    # ============================================
    # Solve
    # ============================================
    optimize!(model)

    status = termination_status(model)
    if status != MOI.OPTIMAL
        @warn "Optimization not optimal at t=$t. Status: $status"
    end

    # Extract solution
    u_val     = value.(u)
    rho_val   = value(rho)
    delta_val = value(delta)

    return u_val, rho_val, delta_val
end

In [ ]:
## Main simulation for ODCLFCBF with Pointing Constraints (Case 2)
p_rho = 0.1
p_delta = 100.0

# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
rho_hist = zeros(n_t_grid)
delta_hist = zeros(n_t_grid)
x[:, 1] = x0

# Storage for pointing constraint data
n_sensors = length(params.b_sensors)
n_bright_objs = length(params.n_bright)
n_constraints = n_sensors * n_bright_objs
b_vals = zeros(n_constraints, n_t_grid)
angles_deg = zeros(n_constraints, n_t_grid)

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input, decay weight, and slack variable
    u_val, rho_val, delta_val = ODCLFCBF_control_pointing(t_grid[i], x[:, i], R_gain, p_rho, p_delta, params)
    u[:, i] = u_val
    rho_hist[i] = rho_val
    delta_hist[i] = delta_val
    
    # Compute pointing constraint values at current state
    sigma_current = x[1:3, i]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, angle_deg = compute_pointing_constraint(
                sigma_current,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals[idx, i] = b_val
            angles_deg[idx, i] = angle_deg
            idx += 1
        end
    end
    
    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end

# Compute final time step pointing constraints
sigma_final = x[1:3, end]
idx = 1
for s in eachindex(params.b_sensors)
    for j in eachindex(params.n_bright)
        b_val, angle_deg = compute_pointing_constraint(
            sigma_final,
            params.b_sensors[s],
            params.n_bright[j],
            params.theta_fov[s]
        )
        b_vals[idx, end] = b_val
        angles_deg[idx, end] = angle_deg
        idx += 1
    end
end

end_time = time()
elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds")
elapsed_time_ODCLFCBF = elapsed_time

# Convert RPs to Euler angles for visualization
psi_ODCLFCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the OD-CLF-CBF-QP Controller
u_ODCLFCBF = u
x_ODCLFCBF = x
t_ODCLFCBF = collect(t_grid)
rho_ODCLFCBF = rho_hist
delta_ODCLFCBF = delta_hist
b_vals_ODCLFCBF = b_vals
angles_deg_ODCLFCBF = angles_deg

# Cost calculation
Cost_ODCLFCBF = (
    trapz(t_ODCLFCBF, [x_ODCLFCBF[1:6, i]' * Q * x_ODCLFCBF[1:6, i] + 
                        u_ODCLFCBF[:, i]' * R * u_ODCLFCBF[:, i] for i in 1:length(t_grid)])
)
println("Cost J_ODCLFCBF = $Cost_ODCLFCBF")

# Check asymptotic stability via slack variable
if abs(delta_hist[end]) < 1e-6
    println("δ → 0: asymptotic stability assured")
else
    println("δ = $(delta_hist[end]) at final time (not yet converged)")
end

# Check pointing constraint satisfaction
min_b_odclfcbf = minimum(b_vals_ODCLFCBF)
println("ODCLFCBF minimum pointing constraint value: $(round(min_b_odclfcbf, digits=6))")
violation_tol = 1e-3
if min_b_odclfcbf >= -violation_tol
    println("ODCLFCBF pointing constraint satisfied throughout trajectory")
else
    println("ODCLFCBF pointing constraint VIOLATED! (min b = $(round(min_b_odclfcbf, digits=6)))")
end

In [ ]:
## RES-CLF-CBF-QP Controller - Case 2: Pointing Constraints
function RESCLFCBF_control_pointing(t, x, R_gain, epsilon, p_delta, params)
    # Extract state variables
    sigma = x[1:3]
    w = x[4:6]
    hw = x[7:9]

    # Extract parameters
    J = params.J
    J_inv = inv(J)

    # ============================================
    # System Dynamics (matching spacecraft_plant)
    # ============================================
    sigmac = [0.0       -sigma[3]   sigma[2];
              sigma[3]   0.0       -sigma[1];
             -sigma[2]   sigma[1]   0.0     ]

    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')

    wc = [0.0    -w[3]   w[2];
          w[3]    0.0   -w[1];
         -w[2]    w[1]   0.0 ]

    # Drift part of ω̇ (unforced): J⁻¹(-ω×(Jω + hw))
    wdot_drift = J_inv * (-wc * (J * w + hw))

    # ============================================
    # CLF Construction via Input-Output Linearization
    # ============================================
    # Output: y = sigma, relative degree r = 2
    sigmadot = M_sigma * w

    # Ṁ(σ) = (1/2)(σ̇× + σ̇ σᵀ + σ σ̇ᵀ)
    sigmadotc = [0.0          -sigmadot[3]   sigmadot[2];
                 sigmadot[3]   0.0          -sigmadot[1];
                -sigmadot[2]   sigmadot[1]   0.0        ]

    Mdot_sigma = (1/2) * (sigmadotc + sigmadot * sigma' + sigma * sigmadot')

    # Second-order Lie derivatives
    L_f_2_h   = Mdot_sigma * w + M_sigma * wdot_drift
    L_g_L_f_h = M_sigma * J_inv

    # Feedforward term: u* = −(L_g L_f h)⁻¹ L_f² h
    L_mat = inv(L_g_L_f_h)
    ustar = -L_mat * L_f_2_h

    # Riccati equation matrices
    eta = vcat(sigma, sigmadot)

    F = [zeros(3, 3)            Matrix{Float64}(I, 3, 3);
         zeros(3, 3)            zeros(3, 3)             ]

    G_mat = [zeros(3, 3);
             Matrix{Float64}(I, 3, 3)]

    Q = Symmetric(Matrix{Float64}(I, 6, 6))

    R_mat = Symmetric(R_gain * (L_mat' * L_mat))

    # Solve CARE: Fᵀ P + P F + Q − P G R⁻¹ Gᵀ P = 0
    S = G_mat * inv(R_mat) * G_mat'
    S = Symmetric((S + S') / 2)
    P, _, _ = arec(F, S, Q)

    # CLF value and Lie derivatives
    V_val    = dot(eta, P * eta)
    L_fbar_V = dot(eta, (F' * P + P * F) * eta)
    L_gbar_V = 2.0 * eta' * P * G_mat            # 1×3 row vector

    # ============================================
    # RES-CLF decay rate (Eq. 14, Remark 3)
    # W(η) = (1/ε)(λ_min(Q)/λ_max(P)) V(η)
    # ============================================
    lambda_min_Q = minimum(eigvals(Matrix(Q)))
    lambda_max_P = maximum(eigvals(P))
    W_val = (1.0 / epsilon) * (lambda_min_Q / lambda_max_P) * V_val

    # CLF constraint coefficient: L_ḡ V · L_g L_f h
    coeff_clf = vec(L_gbar_V * L_g_L_f_h)         # 3-element vector

    # Objective weighting
    H = Matrix{Float64}(I, 3, 3)
    Hbar = L_g_L_f_h' * H * L_g_L_f_h

    # ============================================
    # Rotation matrix R(σ): inertial → body
    # ============================================
    R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)

    # HOCBF parameters
    alpha1_hocbf = params.alpha1_HOCBF
    alpha2_hocbf = params.alpha2_HOCBF

    # ============================================
    # RES-CLF-QP with HOCBF Pointing
    # ============================================
    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, u[1:3])
    @variable(model, delta)

    # Objective: ‖L̄(u − u*)‖²_H  +  p_δ δ²
    @objective(model, Min,
        sum(Hbar[i, j] * (u[i] - ustar[i]) * (u[j] - ustar[j])
            for i in 1:3, j in 1:3) +
        p_delta * delta^2
    )

    # CLF constraint: L_f̄ V + L_ḡ V · L̄(u − u*) ≤ −W + δ
    @constraint(model,
        L_fbar_V + sum(coeff_clf[i] * (u[i] - ustar[i]) for i in 1:3) <=
        -W_val + delta
    )

    # ============================================
    # Input constraints
    # ============================================
    @constraint(model, u .>= params.umin)
    @constraint(model, u .<= params.umax)

    # ============================================
    # HOCBF pointing constraints
    # ============================================
    for i in eachindex(params.b_sensors)
        b_i = params.b_sensors[i]
        theta_i = params.theta_fov[i]

        for j in eachindex(params.n_bright)
            n_j = params.n_bright[j]

            # Bright object direction in body frame
            r_j = R_sigma * n_j

            # c = rⱼ × bᵢ
            c = cross(r_j, b_i)

            # ψ₀ = cos(θᵢ) - bᵢᵀ rⱼ
            b_val = cos(theta_i) - dot(b_i, r_j)

            # L_F b = cᵀ ω
            Lf_b = dot(c, w)

            # ċ = ṙⱼ × bᵢ = (-wc * rⱼ) × bᵢ = bᵢ × (ω × rⱼ)
            c_dot = cross(b_i, wc * r_j)

            # L²_F b = ċᵀω + cᵀ(J⁻¹(-ω×(Jω + hw)))
            Lf2_b = dot(c_dot, w) + dot(c, wdot_drift)

            # L_G L_F b = cᵀ J⁻¹   (3-vector, coefficients of u)
            LgLf_b = c' * J_inv

            # HOCBF constraint: L²b + LgLf·u + (α₁+α₂)Lfb + α₁α₂·b ≥ 0
            @constraint(model,
                Lf2_b + LgLf_b * u + (alpha1_hocbf + alpha2_hocbf) * Lf_b +
                alpha1_hocbf * alpha2_hocbf * b_val >= 0
            )
        end
    end

    # ============================================
    # Solve
    # ============================================
    optimize!(model)

    status = termination_status(model)
    if status != MOI.OPTIMAL
        @warn "Optimization not optimal at t=$t. Status: $status"
    end

    # Extract solution
    u_val     = value.(u)
    delta_val = value(delta)

    return u_val, delta_val
end

In [ ]:
## Main simulation for RESCLFCBF with Pointing Constraints (Case 2)
R_gain = 10.0
epsilon = 0.02
p_delta = 100.0

# Initialize time and state vectors
dt = 0.1
t_grid = range(t_span[1], t_span[2], step=dt)
n_t_grid = length(t_grid)
u = zeros(3, n_t_grid)
x = zeros(9, n_t_grid)
delta_hist = zeros(n_t_grid)
x[:, 1] = x0

# Storage for pointing constraint data
n_sensors = length(params.b_sensors)
n_bright_objs = length(params.n_bright)
n_constraints = n_sensors * n_bright_objs
b_vals = zeros(n_constraints, n_t_grid)
angles_deg = zeros(n_constraints, n_t_grid)

start_time = time()
for i in 1:(n_t_grid-1)
    # Get the control input and slack variable (no rho)
    u_val, delta_val = RESCLFCBF_control_pointing(t_grid[i], x[:, i], R_gain, epsilon, p_delta, params)
    u[:, i] = u_val
    delta_hist[i] = delta_val
    
    # Compute pointing constraint values at current state
    sigma_current = x[1:3, i]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, angle_deg = compute_pointing_constraint(
                sigma_current,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals[idx, i] = b_val
            angles_deg[idx, i] = angle_deg
            idx += 1
        end
    end
    
    # Solve the ODE between t[i] and t[i+1] using u[:, i]
    t_temp = (t_grid[i], t_grid[i] + dt)
    dynamics = spacecraft_plant_closure(u[:, i], params)
    prob = ODEProblem(dynamics, x[:, i], t_temp)
    sol = solve(prob, RK4(), saveat=t_temp, reltol=1e-4, abstol=1e-6)
    x[:, i+1] = sol.u[end]
end

# Compute final time step pointing constraints
sigma_final = x[1:3, end]
idx = 1
for s in eachindex(params.b_sensors)
    for j in eachindex(params.n_bright)
        b_val, angle_deg = compute_pointing_constraint(
            sigma_final,
            params.b_sensors[s],
            params.n_bright[j],
            params.theta_fov[s]
        )
        b_vals[idx, end] = b_val
        angles_deg[idx, end] = angle_deg
        idx += 1
    end
end

end_time = time()
elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds")
elapsed_time_RESCLFCBF = elapsed_time

# Convert RPs to Euler angles for visualization
psi_RESCLFCBF = hcat([rps_to_euler(x[1:3, i]) * (180 / π) for i in 1:n_t_grid]...)'

# Data for the RES-CLF-CBF-QP Controller
u_RESCLFCBF = u
x_RESCLFCBF = x
t_RESCLFCBF = collect(t_grid)
delta_RESCLFCBF = delta_hist
b_vals_RESCLFCBF = b_vals
angles_deg_RESCLFCBF = angles_deg

# Cost calculation
Cost_RESCLFCBF = (
    trapz(t_RESCLFCBF, [x_RESCLFCBF[1:6, i]' * Q * x_RESCLFCBF[1:6, i] + 
                        u_RESCLFCBF[:, i]' * R * u_RESCLFCBF[:, i] for i in 1:length(t_grid)])
)
println("Cost J_RESCLFCBF = $Cost_RESCLFCBF")

# Check asymptotic stability via slack variable
if abs(delta_hist[end]) < 1e-6
    println("δ → 0: asymptotic stability assured")
else
    println("δ = $(delta_hist[end]) at final time (not yet converged)")
end

# Check pointing constraint satisfaction
min_b_resclfcbf = minimum(b_vals_RESCLFCBF)
println("RESCLFCBF minimum pointing constraint value: $(round(min_b_resclfcbf, digits=6))")
violation_tol = 1e-3
if min_b_resclfcbf >= -violation_tol
    println("RESCLFCBF pointing constraint satisfied throughout trajectory")
else
    println("RESCLFCBF pointing constraint VIOLATED! (min b = $(round(min_b_resclfcbf, digits=6)))")
end

In [ ]:
## ==========================================================================
## Plot Trajectories — Spacecraft (Case 2: Pointing Constraints)
## ==========================================================================
# Requires: 
#           x_RESCLFCBF, u_RESCLFCBF, t_RESCLFCBF                (RES-CLF-CBF-QP)
#           x_ODCLFCBF, u_ODCLFCBF, t_ODCLFCBF                   (OD-CLF-CBF-QP)
#           x_OCP, u_OCP, t_OCP                                  (Optimal Control)
#           x_GHJBCBF, u_GHJBCBF, t_GHJBCBF                      (GHJB-CBF-QP)

mkpath("Example_2_Figures")

## Plot Trajectories (σ, ω, hw, u)
selected_controllers = ["1", "2", "3", "4"]
input_list = ["1", "2", "3", "4"]
# ============================================
# Compute pointing constraint values FIRST
# ============================================
n_t_ocp = length(t_OCP)
n_constraints = length(params.b_sensors) * length(params.n_bright)
b_vals_OCP = zeros(n_constraints, n_t_ocp)
angles_deg_OCP = zeros(n_constraints, n_t_ocp)
for i in 1:n_t_ocp
    sigma_current = x_OCP[1:3, i]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, angle_deg = compute_pointing_constraint(
                sigma_current,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals_OCP[idx, i] = b_val
            angles_deg_OCP[idx, i] = angle_deg
            idx += 1
        end
    end
end
# Check constraint satisfaction
min_b_ocp = minimum(b_vals_OCP)
println("OCP minimum pointing constraint value: ", min_b_ocp)
violation_tol = 1e-3
if min_b_ocp >= - violation_tol
    println("OCP pointing constraint satisfied throughout trajectory")
else
    println("OCP pointing constraint VIOLATED! (min b = $min_b_ocp)")
end
# ============================================
# Now build data dict with correct b_vals
# ============================================
controllers_data = Dict(
    "1" => Dict(
        "label" => "RES-CLF-CBF-QP", 
        "color" => "tab:green", 
        "x" => x_RESCLFCBF, 
        "u" => u_RESCLFCBF, 
        "t" => t_RESCLFCBF,
        "b_vals" => b_vals_RESCLFCBF,
        "angles_deg" => angles_deg_RESCLFCBF
    ),
    "2" => Dict(
        "label" => "OD-CLF-CBF-QP", 
        "color" => "tab:orange", 
        "x" => x_ODCLFCBF, 
        "u" => u_ODCLFCBF, 
        "t" => t_ODCLFCBF,
        "b_vals" => b_vals_ODCLFCBF,
        "angles_deg" => angles_deg_ODCLFCBF
    ),
    "3" => Dict(
        "label" => "Open-Loop Optimal (OCP)", 
        "color" => "tab:blue", 
        "x" => x_OCP, 
        "u" => u_OCP, 
        "t" => t_OCP,
        "b_vals" => b_vals_OCP,
        "angles_deg" => angles_deg_OCP
    ),
    "4" => Dict(
        "label" => "GHJB-CBF-QP", 
        "color" => "black", 
        "x" => x_GHJBCBF, 
        "u" => u_GHJBCBF, 
        "t" => t_GHJBCBF,
        "b_vals" => b_vals_GHJBCBF,
        "angles_deg" => angles_deg_GHJBCBF
    )
)
# Plot state trajectories (σ, ω, hw, u) for each axis
for i in 1:3
    fig, (ax1, ax2, ax3, ax4) = subplots(4, figsize=(3.41, 4.0), sharex=true)
    fig.set_constrained_layout(true)
    
    for controller in selected_controllers
        if controller in input_list
            data = controllers_data[controller]
            ax1.plot(data["t"], data["x"][i, :], linestyle="solid", color=data["color"], zorder=3, label=data["label"])
            ax2.plot(data["t"], data["x"][i+3, :], linestyle="solid", color=data["color"], zorder=3)
            ax3.plot(data["t"], data["x"][i+6, :], linestyle="solid", color=data["color"], zorder=3)
            ax4.plot(data["t"], data["u"][i, :], linestyle="solid", color=data["color"], zorder=3)
        end
    end
    
    ax1.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax1.set_ylabel(L"\sigma_{%$(i)}")
    
    ax2.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax2.set_ylabel(L"\omega_{%$(i)} \text{ (rad/s)}")
    
    ax3.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax3.set_ylabel(L"h_{w%$(i)} \text{ (N m s)}")
    
    ax4.hlines(y=0, xmin=0, xmax=t_grid[end], color="grey", linestyle="dotted", zorder=2)
    ax4.hlines(y=params.umax, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2, label="Input Constraints")
    ax4.hlines(y=params.umin, xmin=0, xmax=t_grid[end], color="red", linestyle="dashed", zorder=2)
    ax4.set_ylabel(L"u_{%$(i)} \text{ (N m)}")
    # ============================================
    # Fill to padded plot edges
    # ============================================
    yl4 = ax4.get_ylim()
    pad4 = 0.05 * (yl4[2] - yl4[1])  # 5% padding — adjust to taste
    new_yl4 = (min(yl4[1], params.umin[i]) - pad4, max(yl4[2], params.umax[i]) + pad4)
    ax4.fill_between(t_GHJBCBF, params.umax[i], new_yl4[2], alpha=0.2, color="red")
    ax4.fill_between(t_GHJBCBF, new_yl4[1], params.umin[i], alpha=0.2, color="red")
    ax4.set_ylim(new_yl4)
    # ============================================
    
    for ax in [ax1, ax2, ax3, ax4]
        ax.autoscale(enable=true, axis="x", tight=true)
        ax.grid(true, zorder=0, color="0.95", linestyle="dotted", which="both", axis="both")
    end
    
    ax4.set_xlabel("Time (s)")
    
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles3, labels3 = ax4.get_legend_handles_labels()
    handles = vcat(handles1, handles3)
    labels = vcat(labels1, labels3)
    
    if length(handles) > 0
        fig.legend(handles, labels, loc="outside upper center", ncol=2, framealpha=0, edgecolor="none", facecolor="white").set_zorder(2)
    end
    
    # show()
    savefig("Example_2_Figures/trajectories_spacecraft_case2_$(i).pdf", dpi=300, bbox_inches="tight", pad_inches=0.0)
    println("Saved: Example_2_Figures/trajectories_spacecraft_case2_$(i).pdf")
end

## Plot Pointing Constraint Value Over Time
# Compute steady-state B(σ) at σ=0
b_steady_state = cos(params.theta_fov[1]) - dot(params.b_sensors[1], params.n_bright[1])
fig, ax = subplots(1, figsize=(3.41, 1.7))
fig.set_constrained_layout(true)
for controller in selected_controllers
    if controller in input_list
        data = controllers_data[controller]
        min_b_over_time = [minimum(data["b_vals"][:, i]) for i in 1:size(data["b_vals"], 2)]
        ax.plot(data["t"], min_b_over_time, linestyle="solid", color=data["color"], zorder=3, label=data["label"])
    end
end
ax.axhline(y=0, color="red", linestyle="dashed", label=L"$B(\sigma)=0$")
ax.axhline(y=b_steady_state, color="grey", linestyle="dotted", zorder=2, label=L"$B(\sigma=0) \approx $" * string(round(b_steady_state, digits=3)) * "")
ax.fill_between(t_GHJBCBF, -0.05, 0, alpha=0.2, color="red")
ax.set_ylabel(L"B(\sigma)")
ax.set_xlabel("Time (s)")
ax.autoscale(enable=true, axis="x", tight=true)
ax.grid(true, zorder=0, color="0.95", linestyle="dotted", which="both", axis="both")
y_max = maximum([maximum(controllers_data[c]["b_vals"]) for c in selected_controllers if c in input_list])
ax.set_ylim(-0.05, y_max * 1.1)
handles, labels = ax.get_legend_handles_labels()
if length(handles) > 0
    fig.legend(handles, labels, loc="outside upper center", ncol=2, framealpha=0, edgecolor="none", facecolor="white").set_zorder(2)
end

# show()
savefig("Example_2_Figures/pointing_constraint_spacecraft_case2.pdf", dpi=300, bbox_inches="tight", pad_inches=0.0)
println("Saved: Example_2_Figures/pointing_constraint_spacecraft_case2.pdf")

In [ ]:
## Spacecraft Simulation Using Nonlinear Optimal Control (UNCONSTRAINED)
# Problem parameters
xf = zeros(9)           # Desired final state [sigma; w; hw] all zeros
uf = zeros(3)           # Desired final control
dt = 0.1
n_steps = Int(t_span[2] / dt)

# Time grid (linear)
timepts = range(0, t_span[2], length=n_steps+1)

# Setup JuMP OCP
model = Model(Ipopt.Optimizer)
set_optimizer_attribute(model, "max_iter", 1000)
set_optimizer_attribute(model, "tol", 1e-10)
set_optimizer_attribute(model, "mu_strategy", "adaptive")
set_optimizer_attribute(model, "print_level", 1)

# Variables
@variable(model, X[1:9, 1:n_steps+1])      # [sigma; w; hw]
@variable(model, U[1:3, 1:n_steps+1])      # Control torques

# ============================================
# Helper: symbolic dynamics xdot = f(x) + g(x)*u for JuMP variables
# ============================================
J_inv_num = inv(params.J)

function ocp_xdot(sigma, w, hw, u)
    sigmac = [ 0.0       -sigma[3]  sigma[2]; 
               sigma[3]   0.0      -sigma[1]; 
              -sigma[2]   sigma[1]  0.0     ]
    
    M_sigma = (1/2) * (I(3) + sigmac + sigma * sigma')
    
    wc = [ 0.0    -w[3]   w[2]; 
           w[3]    0.0   -w[1]; 
          -w[2]    w[1]   0.0 ]
    
    sigma_dot = M_sigma * w
    w_dot = J_inv_num * (-wc * (params.J * w + hw)) + J_inv_num * u
    hw_dot = -u
    
    return vcat(sigma_dot, w_dot, hw_dot)
end

# ============================================
# Explicit Midpoint (RK2) Dynamics Constraints
# ============================================
for t in 1:n_steps
    x_t = X[:, t]
    u_t = U[:, t]
    
    # Stage 1: evaluate at current state
    k1 = ocp_xdot(x_t[1:3], x_t[4:6], x_t[7:9], u_t)
    
    # Stage 2: evaluate at midpoint
    x_mid = x_t + (dt/2) * k1
    k2 = ocp_xdot(x_mid[1:3], x_mid[4:6], x_mid[7:9], u_t)
    
    # Midpoint update
    @constraint(model, X[:, t+1] .== x_t + dt * k2)
end

# Objective with trapezoidal integration
cost = 0
for t in 1:n_steps
    x_diff_t = X[1:6, t] - xf[1:6]
    integrand_t = x_diff_t' * Q * x_diff_t + U[:, t]' * R * U[:, t]
    
    x_diff_tnext = X[1:6, t+1] - xf[1:6]
    integrand_tnext = x_diff_tnext' * Q * x_diff_tnext + U[:, t+1]' * R * U[:, t+1]
    
    cost += (dt/2) * (integrand_t + integrand_tnext)
end

@objective(model, Min, cost)

# Initial condition
@constraint(model, X[:, 1] .== x0)

# Input constraints only (NO pointing constraints)
@constraint(model, [t=1:n_steps+1, i=1:3], params.umin[i] <= U[i, t] <= params.umax[i])

# Solve
start_time = time()
optimize!(model)
end_time = time()

elapsed_time = end_time - start_time
println("T=$(t_span[2]): Elapsed time: $(round(elapsed_time, digits=2)) seconds ($(round(elapsed_time/60, digits=2)) minutes)")
elapsed_time_OCP_unconstrained = elapsed_time

# Extract results
x_OCP_unconstrained = value.(X)
u_OCP_unconstrained = value.(U)
t_OCP_unconstrained = collect(range(0, t_span[2], length=n_steps+1))
Cost_OCP_unconstrained = objective_value(model)
println("Cost_OCP_unconstrained: $Cost_OCP_unconstrained")
println("Optimization status: ", termination_status(model))

# Convert RPs to Euler angles
psi_OCP_unconstrained = hcat([rps_to_euler(x_OCP_unconstrained[1:3, i]) * (180 / π) for i in 1:size(x_OCP_unconstrained, 2)]...)'

# Check if unconstrained trajectory violates pointing constraints
n_constraints = length(params.b_sensors) * length(params.n_bright)
b_vals_unconstrained = zeros(n_constraints, n_steps+1)
for i in 1:n_steps+1
    sigma_current = x_OCP_unconstrained[1:3, i]
    idx = 1
    for s in eachindex(params.b_sensors)
        for j in eachindex(params.n_bright)
            b_val, _ = compute_pointing_constraint(
                sigma_current,
                params.b_sensors[s],
                params.n_bright[j],
                params.theta_fov[s]
            )
            b_vals_unconstrained[idx, i] = b_val
            idx += 1
        end
    end
end
min_b_unconstrained = minimum(b_vals_unconstrained)
println("Unconstrained OCP minimum pointing constraint value: $(round(min_b_unconstrained, digits=6))")
if min_b_unconstrained >= -1e-3
    println("Unconstrained OCP happens to satisfy pointing constraints")
else
    println("Unconstrained OCP violates pointing constraints (as expected, min b = $(round(min_b_unconstrained, digits=6)))")
end

In [ ]:
## ====================================================================================
## 3D Sphere Visualization of Pointing Constraint (All Controllers + Unconstrained OCP)
## ====================================================================================
# Requires: 
#           x_RESCLFCBF, u_RESCLFCBF, t_RESCLFCBF                          (RES-CLF-CBF-QP)
#           x_ODCLFCBF, u_ODCLFCBF, t_ODCLFCBF                             (OD-CLF-CBF-QP)
#           x_OCP, u_OCP, t_OCP                                            (Optimal Control)
#           x_GHJBCBF, u_GHJBCBF, t_GHJBCBF                                (GHJB-CBF-QP)
#           x_OCP_unconstrained, u_OCP_unconstrained, t_OCP_unconstrained  (Unconstrained OCP)

mkpath("Example_2_Figures")

function plot_pointing_sphere(x_GHJBCBF, x_ODCLFCBF, x_RESCLFCBF, x_OCP, x_OCP_uncon, params; save_name="Example_2_Figures/pointing_sphere_3d.pdf")
    
    b_sensor = params.b_sensors[1]
    n_bright = params.n_bright[1]
    theta_fov = params.theta_fov[1]
    
    # ============================================
    # Compute sensor boresight in inertial frame
    # R(σ) maps inertial → body, so R(σ)ᵀ maps body → inertial
    # sensor_inertial = R(σ)ᵀ * b_sensor
    # ============================================
    function compute_sensor_inertial(x_traj)
        n_t = size(x_traj, 2)
        sensor_dirs = zeros(3, n_t)
        for i in 1:n_t
            sigma = x_traj[1:3, i]
            
            sigmac = [ 0.0       -sigma[3]  sigma[2]; 
                       sigma[3]   0.0      -sigma[1]; 
                      -sigma[2]   sigma[1]  0.0     ]
            R_sigma = (1 / (1 + sigma' * sigma)) * ((1 - sigma' * sigma) * I(3) + 2 * (sigma * sigma') - 2 * sigmac)
            sensor_dirs[:, i] = R_sigma' * b_sensor
        end
        return sensor_dirs
    end
    
    sensor_ghjbcf = compute_sensor_inertial(x_GHJBCBF)
    sensor_odclfcbf = compute_sensor_inertial(x_ODCLFCBF)
    sensor_resclfcbf = compute_sensor_inertial(x_RESCLFCBF)
    sensor_ocp = compute_sensor_inertial(x_OCP)
    sensor_ocp_uncon = compute_sensor_inertial(x_OCP_uncon)
    
    # ============================================
    # Build the unit sphere (wireframe)
    # ============================================
    n_sphere = 40
    u_angles = range(0, 2π, length=n_sphere)
    v_angles = range(0, π, length=n_sphere)
    
    x_sphere = cos.(u_angles) * sin.(v_angles)'
    y_sphere = sin.(u_angles) * sin.(v_angles)'
    z_sphere = ones(n_sphere) * cos.(v_angles)'
    
    # ============================================
    # Build the exclusion cone on the sphere surface
    # ============================================
    n_cone_pts = 100
    phi_cone = range(0, 2π, length=n_cone_pts)
    
    if abs(n_bright[1]) < 0.9
        v_temp = [1.0, 0.0, 0.0]
    else
        v_temp = [0.0, 1.0, 0.0]
    end
    e1 = normalize(cross(n_bright, v_temp))
    e2 = normalize(cross(n_bright, e1))
    
    cone_boundary = zeros(3, n_cone_pts)
    for i in 1:n_cone_pts
        cone_boundary[:, i] = cos(theta_fov) * n_bright + sin(theta_fov) * (cos(phi_cone[i]) * e1 + sin(phi_cone[i]) * e2)
    end
    
    # Fill the exclusion zone cap with a mesh
    n_cap_rings = 20
    cap_angles = range(0, theta_fov, length=n_cap_rings)
    x_cap = zeros(n_cap_rings, n_cone_pts)
    y_cap = zeros(n_cap_rings, n_cone_pts)
    z_cap = zeros(n_cap_rings, n_cone_pts)
    
    for (ri, ang) in enumerate(cap_angles)
        for (ci, ph) in enumerate(phi_cone)
            pt = cos(ang) * n_bright + sin(ang) * (cos(ph) * e1 + sin(ph) * e2)
            x_cap[ri, ci] = pt[1]
            y_cap[ri, ci] = pt[2]
            z_cap[ri, ci] = pt[3]
        end
    end
    
    # ============================================
    # Create the 3D figure
    # ============================================
    fig = figure(figsize=(3.41, 3.5))
    ax = fig.add_subplot(111, projection="3d")
    ax.computed_zorder = false    # Force manual zorder (disable depth sorting)
    
    # Shrink sphere slightly so trajectories (on unit sphere) sit above it
    r_sphere = 0.99
    ax.plot_wireframe(r_sphere * x_sphere, r_sphere * y_sphere, r_sphere * z_sphere, 
                      color="gray", alpha=0.4, linewidth=0.2, zorder=0)
    
    # ============================================
    # Plot sensor trajectory (RES-CLF-CBF-QP)
    # ============================================
    ax.plot(sensor_resclfcbf[1, :], sensor_resclfcbf[2, :], sensor_resclfcbf[3, :], 
            "tab:green", linestyle="solid", label="RES-CLF-CBF-QP", zorder=5)
    # ============================================
    # Plot sensor trajectory (OD-CLF-CBF-QP)
    # ============================================
    ax.plot(sensor_odclfcbf[1, :], sensor_odclfcbf[2, :], sensor_odclfcbf[3, :], 
            "tab:orange", linestyle="solid", label="OD-CLF-CBF-QP", zorder=5)
    # ============================================
    # Plot sensor trajectory (Constrained OCP)
    # ============================================
    ax.plot(sensor_ocp[1, :], sensor_ocp[2, :], sensor_ocp[3, :], 
            "tab:blue", linestyle="solid", label="Open-Loop Optimal (OCP)", zorder=5)
    # ============================================
    # Plot sensor trajectory (GHJBCBF)
    # ============================================
    ax.plot(sensor_ghjbcf[1, :], sensor_ghjbcf[2, :], sensor_ghjbcf[3, :], 
            "black", linestyle="solid", label="GHJB-CBF-QP", zorder=5)
    # ============================================
    # Plot sensor trajectory (Unconstrained OCP)
    # ============================================
    ax.plot(sensor_ocp_uncon[1, :], sensor_ocp_uncon[2, :], sensor_ocp_uncon[3, :], 
            "tab:blue", linestyle="dashed", alpha=0.7, label="OCP (Unconstrained)", zorder=4)
    
    # ============================================
    # Plot exclusion zone
    # ============================================
    ax.plot_surface(x_cap, y_cap, z_cap, 
                    color="red", alpha=0.5, edgecolor="none", zorder=2)
    
    ax.plot(cone_boundary[1, :], cone_boundary[2, :], cone_boundary[3, :], 
            "red", linestyle="dashed", label=L"Exclusion zone ($\theta = %$(Int(round(rad2deg(theta_fov))))^{\!\circ}\!$)", zorder=3)
    
    ax.scatter([n_bright[1]], [n_bright[2]], [n_bright[3]], 
               color="red", s=60, marker="*", zorder=6, label="Bright Object (n)")
    
    ax.plot([0, n_bright[1]*1.0], [0, n_bright[2]*1.0], [0, n_bright[3]*1.0], 
            "red", alpha=0.5, zorder=1)
    
    # ============================================
    # Plot start/end points
    # ============================================
    ax.scatter([sensor_ghjbcf[1, 1]], [sensor_ghjbcf[2, 1]], [sensor_ghjbcf[3, 1]], 
               color="green", s=45, marker="o", zorder=3, label="Start")
    ax.scatter([sensor_ghjbcf[1, end]], [sensor_ghjbcf[2, end]], [sensor_ghjbcf[3, end]], 
               color="darkblue", s=40, marker="s", zorder=3, label="End")
    
    # ============================================
    # Plot sensor boresight direction (body frame, fixed)
    # ============================================
    ax.plot([0, b_sensor[1]*0.4], [0, b_sensor[2]*0.4], [0, b_sensor[3]*0.4], 
            "black", linestyle="solid", alpha=0.5, zorder=1)
    ax.text(b_sensor[1]*0.45, b_sensor[2]*0.45, b_sensor[3]*0.45, 
            "b (sensor)", color="black")
    
    # ============================================
    # Formatting: remove axis box so sphere fills the figure
    # ============================================
    ax.set_axis_off()
    
    ax.set_xlim([-1.05, 1.05])
    ax.set_ylim([-1.05, 1.05])
    ax.set_zlim([-1.05, 1.05])
    
    ax.set_box_aspect([1, 1, 1], zoom=1.5)
    
    # Legend
    handles, labels = ax.get_legend_handles_labels()
    if length(handles) > 0
        fig.legend(handles, labels, 
                   bbox_to_anchor=(0.5, 1.15), loc="upper center", 
                   ncol=2, framealpha=0, edgecolor="none", facecolor="white", 
                   handletextpad=0.8, columnspacing=2.0).set_zorder(10)
    end
    
    elev = 45
    azim = -110
    ax.view_init(elev=elev, azim=azim)
    
    # Expand axes to fill figure — shifted up to reduce bottom whitespace
    ax.dist = 9
    ax.set_position([-0.05, -0.08, 1.1, 1.1])
    fig.subplots_adjust(top=1.0, bottom=0.0, left=0.0, right=1.0)
    
    # ============================================
    # Add small coordinate axes triad in bottom-left corner (2D projection)
    # ============================================
    ax_inset = fig.add_axes([0.08, 0.02, 0.15, 0.15])
    ax_inset.set_axis_off()
    ax_inset.set_facecolor("none")
    ax_inset.patch.set_alpha(0.0)
    ax_inset.set_xlim([-0.5, 1.0])
    ax_inset.set_ylim([-0.5, 1.0])
    ax_inset.set_aspect("equal")
    
    # Project 3D unit vectors to 2D using the same view angles
    elev_r = deg2rad(elev)
    azim_r = deg2rad(azim)
    
    function project_3d_to_2d(v)
        x2d = -sin(azim_r)*v[1] + cos(azim_r)*v[2]
        y2d = -cos(elev_r)*cos(azim_r)*v[1] - cos(elev_r)*sin(azim_r)*v[2] + sin(elev_r)*v[3]
        return [x2d, y2d]
    end
    
    dirs_3d = [[1,0,0], [0,1.1,0], [0,0,1.4]]
    labels_xyz = ["X", "Y", "Z"]

    for (d3, lbl) in zip(dirs_3d, labels_xyz)
        d2 = project_3d_to_2d(d3)
        ax_inset.plot([0, d2[1]], [0, d2[2]], color="black", alpha=0.8)
        ax_inset.text(d2[1]*1.3, d2[2]*1.3, lbl,
                      color="black", ha="center", va="center")
    end
    
    # savefig(save_name, dpi=300)
    # show()
    savefig(save_name, dpi=300, bbox_inches="tight", pad_inches=0.0)
    println("Figure saved to $save_name")
    
    return fig, ax
end
# ==============================================
# Call the function with all trajectory matrices
# ==============================================
fig, ax = plot_pointing_sphere(x_GHJBCBF, x_ODCLFCBF, x_RESCLFCBF, x_OCP, x_OCP_unconstrained, params)